In [ ]:
import os
import json
from collections import defaultdict
import math
import matplotlib.pyplot as plt
import numpy as np
import math
import torch
import safetensors.torch
import datasets

## Fine-tuning on NQ with reformulated labels?

In [ ]:
# Define experiment directories for FT_NQ and FT_NQ_RF
FT_NQ_experiments = [
    'll3_8b_0shot', 
    'll3_8b_bm25', 
    'll3_8b_splade',
    'll3_8b_FT_NQ_1epoch_0shot',
    'll3_8b_FT_NQ_1epoch_bm25',
    'll3_8b_FT_NQ_1epoch_splade',
    'll3_8b_FT_NQ_2epochs_0shot',
    'll3_8b_FT_NQ_2epochs_bm25',
    'll3_8b_FT_NQ_2epochs_splade',
]

FT_NQ_RF_labels_experiments = [
    'll3_8b_0shot', 
    'll3_8b_bm25', 
    'll3_8b_splade',
    'll3_8b_FT_NQ_RF_short_1epoch_0shot',
    'll3_8b_FT_NQ_RF_short_1epoch_bm25',
    'll3_8b_FT_NQ_RF_short_1epoch_splade',
    'll3_8b_FT_NQ_RF_short_2epochs_0shot',
    'll3_8b_FT_NQ_RF_short_2epochs_bm25',
    'll3_8b_FT_NQ_RF_short_2epochs_splade',
]

# Root directory where experiment results are stored (4 datasets)
dataset_dirs = [
    'experiments/FT/asqa',
    'experiments/FT/hotpotqa',
    'experiments/FT/bioasq',
    'experiments/FT/syllabusqa'
]

dataset_names = ['ASQA', 'HotpotQA', 'BioASQ', 'SyllabusQA']

# Load metrics for a given experiment directory and dataset
def load_metrics(subdir, dataset_dir):
    metrics_data = {'Match': 0.0, 'Recall': 0.0, 'LLMeval': 0.0}
    json_file_path = os.path.join(dataset_dir, subdir, 'eval_dev_metrics.json')
    if os.path.exists(json_file_path):
        with open(json_file_path, 'r') as json_file:
            metrics = json.load(json_file)
            metrics_data['Match'] = float(metrics.get('M', 0.0))
            metrics_data['Recall'] = float(metrics.get('Recall', 0.0))
            metrics_data['LLMeval'] = float(metrics.get('LLMeval_llama3.1:70b', 0.0))
    return metrics_data

# Plotting function for bar plots with subplots
def plot_metrics_bar_subplots(metric_name, nq_data, nq_rf_data, experiment_labels, title, dataset_names):
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    axs = axs.flatten()

    # Define colors for each dataset pair
    colors = [
        ('#1f77b4', '#ff7f0e'),  # Dataset 1 colors (blue, orange)
        ('#2ca02c', '#d62728'),  # Dataset 2 colors (green, red)
        ('#9467bd', '#8c564b'),  # Dataset 3 colors (purple, brown)
        ('#e377c2', '#7f7f7f'),  # Dataset 4 colors (pink, gray)
    ]

    for i, ax in enumerate(axs):
        bar_width = 0.35
        x = np.arange(len(experiment_labels))
        
        # Plot bars for FT_NQ and FT_NQ_RF for the dataset
        bar1 = ax.bar(x - bar_width/2, nq_data[i], bar_width, label='FT_NQ', color=colors[i][0])
        bar2 = ax.bar(x + bar_width/2, nq_rf_data[i], bar_width, label='FT_NQ_RF', color=colors[i][1])
        
        # Add values on top of bars
        for rect in bar1:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)
        
        for rect in bar2:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

        # increase y_max to fit vertical numerical values
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        new_y_max = y_max + 0.4 * y_range
        ax.set_ylim([y_min, new_y_max])

        
        # Configure plot for the dataset
        ax.set_title(dataset_names[i])
        ax.set_xticks(x)
        ax.set_xticklabels(experiment_labels, rotation=45, ha="right")
        ax.set_ylabel(metric_name)
        ax.legend()

    fig.suptitle(title)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
    plt.savefig(f"tmp/figs/{title.replace(' ', '_').replace(':', '_')}.png")
    plt.show()

# Collect data for both NQ and NQ with Reformulated labels per dataset
def collect_experiment_data_per_dataset(experiments, dataset_dirs):
    matches, recalls, llmevals = [], [], []
    for dataset_dir in dataset_dirs:
        dataset_matches, dataset_recalls, dataset_llmevals = [], [], []
        for experiment in experiments:
            metrics = load_metrics(experiment, dataset_dir)
            dataset_matches.append(metrics['Match'])
            dataset_recalls.append(metrics['Recall'])
            dataset_llmevals.append(metrics['LLMeval'])
        matches.append(dataset_matches)
        recalls.append(dataset_recalls)
        llmevals.append(dataset_llmevals)
    return matches, recalls, llmevals

# Collect data for FT_NQ and FT_NQ_RF experiments
nq_matches, nq_recalls, nq_llmevals = collect_experiment_data_per_dataset(FT_NQ_experiments, dataset_dirs)
nq_rf_matches, nq_rf_recalls, nq_rf_llmevals = collect_experiment_data_per_dataset(FT_NQ_RF_labels_experiments, dataset_dirs)

# Define experiment labels (using only the name part after 'll3_8b_')
experiment_labels = [exp[7:] for exp in FT_NQ_experiments]

# Plot the results using bar plots with subplots for 4 datasets
plot_metrics_bar_subplots('Match', nq_matches, nq_rf_matches, experiment_labels, 'Match Performance: FT_NQ vs FT_NQ_RF', dataset_names)
plot_metrics_bar_subplots('Recall', nq_recalls, nq_rf_recalls, experiment_labels, 'Recall Performance: FT_NQ vs FT_NQ_RF', dataset_names)
plot_metrics_bar_subplots('LLMeval', nq_llmevals, nq_rf_llmevals, experiment_labels, 'LLMeval Performance: FT_NQ vs FT_NQ_RF', dataset_names)


## Better fine-tune on NQ or MultiQA ?

In [ ]:
# Experiments lists
FT_NQ_experiments = [
    'll3_8b_0shot', 
    'll3_8b_bm25', 
    'll3_8b_splade',
    'll3_8b_FT_NQ_1epoch_0shot',
    'll3_8b_FT_NQ_1epoch_bm25',
    'll3_8b_FT_NQ_1epoch_splade',
    'll3_8b_FT_NQ_2epochs_0shot',
    'll3_8b_FT_NQ_2epochs_bm25',
    'll3_8b_FT_NQ_2epochs_splade',
]

FT_NQ_RF_labels_experiments = [
    'll3_8b_0shot', 
    'll3_8b_bm25', 
    'll3_8b_splade',
    'll3_8b_FT_NQ_RF_short_1epoch_0shot',
    'll3_8b_FT_NQ_RF_short_1epoch_bm25',
    'll3_8b_FT_NQ_RF_short_1epoch_splade',
    'll3_8b_FT_NQ_RF_short_2epochs_0shot',
    'll3_8b_FT_NQ_RF_short_2epochs_bm25',
    'll3_8b_FT_NQ_RF_short_2epochs_splade',
]

FT_MQA_experiments = [
    'll3_8b_0shot', 
    'll3_8b_bm25', 
    'll3_8b_splade',
    'll3_8b_FT_MQA_BM25_5K_0shot',
    'll3_8b_FT_MQA_BM25_5K_bm25',
    'll3_8b_FT_MQA_BM25_5K_splade',
    'll3_8b_FT_MQA_BM25_11K_0shot',
    'll3_8b_FT_MQA_BM25_11K_bm25',
    'll3_8b_FT_MQA_BM25_11K_splade',
]

# Dataset names
dataset_names = ['ASQA', 'HotpotQA', 'BioASQ', 'SyllabusQA']

# Load metrics function
def load_metrics(subdir, dataset_dir):
    metrics_data = {'Match': 0.0, 'Recall': 0.0, 'LLMeval': 0.0}
    json_file_path = os.path.join(dataset_dir, subdir, 'eval_dev_metrics.json')
    if os.path.exists(json_file_path):
        with open(json_file_path, 'r') as json_file:
            metrics = json.load(json_file)
            metrics_data['Match'] = float(metrics.get('M', 0.0))
            metrics_data['Recall'] = float(metrics.get('Recall', 0.0))
            metrics_data['LLMeval'] = float(metrics.get('LLMeval_llama3.1:70b', 0.0))
    return metrics_data

# Function for collecting data
def collect_experiment_data_per_dataset(experiments, dataset_dirs):
    matches, recalls, llmevals = [], [], []
    for dataset_dir in dataset_dirs:
        dataset_matches, dataset_recalls, dataset_llmevals = [], [], []
        for experiment in experiments:
            metrics = load_metrics(experiment, dataset_dir)
            dataset_matches.append(metrics['Match'])
            dataset_recalls.append(metrics['Recall'])
            dataset_llmevals.append(metrics['LLMeval'])
        matches.append(dataset_matches)
        recalls.append(dataset_recalls)
        llmevals.append(dataset_llmevals)
    return matches, recalls, llmevals

# Collect data for FT_NQ, FT_NQ_RF, and FT_MQA experiments
nq_matches, nq_recalls, nq_llmevals = collect_experiment_data_per_dataset(FT_NQ_experiments, dataset_dirs)
nq_rf_matches, nq_rf_recalls, nq_rf_llmevals = collect_experiment_data_per_dataset(FT_NQ_RF_labels_experiments, dataset_dirs)
mqa_matches, mqa_recalls, mqa_llmevals = collect_experiment_data_per_dataset(FT_MQA_experiments, dataset_dirs)

# Plotting function for bar plots with subplots
def plot_metrics_bar_subplots(metric_name, nq_data, nq_rf_data, mqa_data, experiment_labels, title, dataset_names):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs = axs.flatten()

    # Define colors for each dataset triplet
    colors = [
        ('#1f77b4', '#ff7f0e', '#2ca02c'),  # Dataset 1 colors (blue, orange, green)
        ('#d62728', '#9467bd', '#8c564b'),  # Dataset 2 colors (red, purple, brown)
        ('#e377c2', '#7f7f7f', '#bcbd22'),  # Dataset 3 colors (pink, gray, yellow)
        ('#17becf', '#f7b6d2', '#aec7e8'),  # Dataset 4 colors (cyan, light pink, light blue)
    ]

    for i, ax in enumerate(axs):
        bar_width = 0.25
        x = np.arange(len(experiment_labels))
        
        # Plot bars for FT_NQ, FT_NQ_RF, and FT_MQA for the dataset
        bar1 = ax.bar(x - bar_width, nq_data[i], bar_width, label='FT_NQ', color=colors[i][0])
        bar2 = ax.bar(x, nq_rf_data[i], bar_width, label='FT_NQ_RF', color=colors[i][1])
        bar3 = ax.bar(x + bar_width, mqa_data[i], bar_width, label='FT_MQA', color=colors[i][2])
        
        # Add values on top of bars
        for rect in bar1:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)
        
        for rect in bar2:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

        for rect in bar3:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

        # Adjust y-axis limits to fit numerical values
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        new_y_max = y_max + 0.4 * y_range
        ax.set_ylim([y_min, new_y_max])

        # Configure plot for the dataset
        ax.set_title(dataset_names[i])
        ax.set_xticks(x)
        ax.set_xticklabels(experiment_labels, rotation=45, ha="right")
        ax.set_ylabel(metric_name)
        ax.legend()

    fig.suptitle(title)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
    plt.savefig(f"tmp/figs/{title.replace(' ', '_').replace(':', '_')}.png")
    plt.show()

# Define experiment labels (shortened)
experiment_labels = ["0 epoch 0 shot", "0 epoch BM25", "0 epoch splade"] + [exp[13:] for exp in FT_NQ_experiments[3:]]

# Plot the results using bar plots with subplots for 4 datasets
plot_metrics_bar_subplots('Match', nq_matches, nq_rf_matches, mqa_matches, experiment_labels, 'Match Performance: FT_NQ vs FT_NQ_RF vs FT_MQA', dataset_names)
plot_metrics_bar_subplots('Recall', nq_recalls, nq_rf_recalls, mqa_recalls, experiment_labels, 'Recall Performance: FT_NQ vs FT_NQ_RF vs FT_MQA', dataset_names)
plot_metrics_bar_subplots('LLMeval', nq_llmevals, nq_rf_llmevals, mqa_llmevals, experiment_labels, 'LLMeval Performance: FT_NQ vs FT_NQ_RF vs FT_MQA', dataset_names)


## Fine-tuning with or without distractors?

In [ ]:
FT_MQA_experiments = [
    'll3_8b_0shot', 
    'll3_8b_bm25', 
    'll3_8b_splade',
    'll3_8b_FT_MQA_BM25_5K_0shot',
    'll3_8b_FT_MQA_BM25_5K_bm25',
    'll3_8b_FT_MQA_BM25_5K_splade',
    'll3_8b_FT_MQA_BM25_11K_0shot',
    'll3_8b_FT_MQA_BM25_11K_bm25',
    'll3_8b_FT_MQA_BM25_11K_splade',
    ]

FT_MQA_with_soft_distractors_experiments = [
    'll3_8b_0shot', 
    'll3_8b_bm25', 
    'll3_8b_splade',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_09_0shot',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_09_bm25',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_09_splade',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_09_0shot',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_09_bm25',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_09_splade',
]

FT_MQA_with_hard_distractors_experiments = [
    'll3_8b_0shot', 
    'll3_8b_bm25', 
    'll3_8b_splade',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_06_0shot',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_06_bm25',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_06_splade',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_06_0shot',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_06_bm25',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_06_splade',
]

# Dataset names
dataset_names = ['ASQA', 'HotpotQA', 'BioASQ', 'SyllabusQA']

# Load metrics function
def load_metrics(subdir, dataset_dir):
    metrics_data = {'Match': 0.0, 'Recall': 0.0, 'LLMeval': 0.0}
    json_file_path = os.path.join(dataset_dir, subdir, 'eval_dev_metrics.json')
    if os.path.exists(json_file_path):
        with open(json_file_path, 'r') as json_file:
            metrics = json.load(json_file)
            metrics_data['Match'] = float(metrics.get('M', 0.0))
            metrics_data['Recall'] = float(metrics.get('Recall', 0.0))
            metrics_data['LLMeval'] = float(metrics.get('LLMeval_llama3.1:70b', 0.0))
    return metrics_data

# Function for collecting data
def collect_experiment_data_per_dataset(experiments, dataset_dirs):
    matches, recalls, llmevals = [], [], []
    for dataset_dir in dataset_dirs:
        dataset_matches, dataset_recalls, dataset_llmevals = [], [], []
        for experiment in experiments:
            metrics = load_metrics(experiment, dataset_dir)
            dataset_matches.append(metrics['Match'])
            dataset_recalls.append(metrics['Recall'])
            dataset_llmevals.append(metrics['LLMeval'])
        matches.append(dataset_matches)
        recalls.append(dataset_recalls)
        llmevals.append(dataset_llmevals)
    return matches, recalls, llmevals

mqa_matches, mqa_recalls, mqa_llmevals = collect_experiment_data_per_dataset(FT_MQA_experiments, dataset_dirs)
mqa_with_soft_distractors_matches, mqa_with_soft_distractors_recalls, mqa_with_soft_distractors_llmevals = collect_experiment_data_per_dataset(FT_MQA_with_soft_distractors_experiments, dataset_dirs)
mqa_with_hard_distractors_matches, mqa_with_hard_distractors_recalls, mqa_with_hard_distractors_llmevals = collect_experiment_data_per_dataset(FT_MQA_with_hard_distractors_experiments, dataset_dirs)

# Plotting function for bar plots with subplots
def plot_metrics_bar_subplots(metric_name, nq_data, nq_rf_data, mqa_data, experiment_labels, title, dataset_names):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs = axs.flatten()

    # Define colors for each dataset triplet
    colors = [
        ('#1f77b4', '#ff7f0e', '#2ca02c'),  # Dataset 1 colors (blue, orange, green)
        ('#d62728', '#9467bd', '#8c564b'),  # Dataset 2 colors (red, purple, brown)
        ('#e377c2', '#7f7f7f', '#bcbd22'),  # Dataset 3 colors (pink, gray, yellow)
        ('#17becf', '#f7b6d2', '#aec7e8'),  # Dataset 4 colors (cyan, light pink, light blue)
    ]

    for i, ax in enumerate(axs):
        bar_width = 0.25
        x = np.arange(len(experiment_labels))
        
        # Plot bars for FT_NQ, FT_NQ_RF, and FT_MQA for the dataset
        bar1 = ax.bar(x - bar_width, nq_data[i], bar_width, label='FT_MQA', color=colors[i][0])
        bar2 = ax.bar(x, nq_rf_data[i], bar_width, label='FT_MQA_soft_distractors', color=colors[i][1])
        bar3 = ax.bar(x + bar_width, mqa_data[i], bar_width, label='FT_MQA_hard_distractors', color=colors[i][2])
        
        # Add values on top of bars
        for rect in bar1:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)
        
        for rect in bar2:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

        for rect in bar3:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

        # Adjust y-axis limits to fit numerical values
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        new_y_max = y_max + 0.4 * y_range
        ax.set_ylim([y_min, new_y_max])

        # Configure plot for the dataset
        ax.set_title(dataset_names[i])
        ax.set_xticks(x)
        ax.set_xticklabels(experiment_labels, rotation=45, ha="right")
        ax.set_ylabel(metric_name)
        ax.legend()

    fig.suptitle(title)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
    plt.savefig(f"tmp/figs/{title.replace(' ', '_').replace(':', '_')}.png")
    plt.show()

# Define experiment labels (shortened)
experiment_labels = ["0 epoch 0 shot", "0 epoch BM25", "0 epoch splade"] +\
      ["1 epoch 0 shot", "1 epoch BM25", "1 epoch splade"] +\
      ["2 epochs 0 shot", "2 epochs BM25", "2 epochs splade"]

# Plot the results using bar plots with subplots for 4 datasets
plot_metrics_bar_subplots('Match', mqa_matches, mqa_with_soft_distractors_matches, mqa_with_hard_distractors_matches, experiment_labels, 'Match Performance: FT_MQA vs FT_MQA_with_soft_distractors vs FT_MQA_with_hard_distractors', dataset_names)
plot_metrics_bar_subplots('Recall', mqa_recalls, mqa_with_soft_distractors_recalls, mqa_with_hard_distractors_recalls, experiment_labels, 'Recall Performance: FT_MQA vs FT_MQA_with_soft_distractors vs FT_MQA_with_hard_distractors', dataset_names)
plot_metrics_bar_subplots('LLMeval', mqa_llmevals, mqa_with_soft_distractors_llmevals, mqa_with_hard_distractors_llmevals, experiment_labels, 'LLMeval Performance: FT_MQA vs FT_MQA_with_soft_distractors vs FT_MQA_with_hard_distractors', dataset_names)


## Fine-tuning on bm25 or on splade?

In [ ]:
FT_bm25 = [
    'll3_8b_FT_MQA_BM25_5K_0shot',
    'll3_8b_FT_MQA_BM25_5K_bm25',
    'll3_8b_FT_MQA_BM25_5K_splade',
    'll3_8b_FT_MQA_BM25_11K_0shot',
    'll3_8b_FT_MQA_BM25_11K_bm25',
    'll3_8b_FT_MQA_BM25_11K_splade',
    ]

FT_splade = [
    'll3_8b_FT_MQA_splade_5K_0shot',
    'll3_8b_FT_MQA_splade_5K_bm25',
    'll3_8b_FT_MQA_splade_5K_splade',
    'll3_8b_FT_MQA_splade_11K_0shot',
    'll3_8b_FT_MQA_splade_11K_bm25',
    'll3_8b_FT_MQA_splade_11K_splade',
    ]

# Root directory where experiment results are stored (4 datasets)
dataset_dirs = [
    'experiments/FT/asqa',
    'experiments/FT/hotpotqa',
    'experiments/FT/bioasq',
    'experiments/FT/syllabusqa'
]

dataset_names = ['ASQA', 'HotpotQA', 'BioASQ', 'SyllabusQA']

# Load metrics for a given experiment directory and dataset
def load_metrics(subdir, dataset_dir):
    metrics_data = {'Match': 0.0, 'Recall': 0.0, 'LLMeval': 0.0}
    json_file_path = os.path.join(dataset_dir, subdir, 'eval_dev_metrics.json')
    if os.path.exists(json_file_path):
        with open(json_file_path, 'r') as json_file:
            metrics = json.load(json_file)
            metrics_data['Match'] = float(metrics.get('M', 0.0))
            metrics_data['Recall'] = float(metrics.get('Recall', 0.0))
            metrics_data['LLMeval'] = float(metrics.get('LLMeval_llama3.1:70b', 0.0))
    return metrics_data

# Plotting function for bar plots with subplots
def plot_metrics_bar_subplots(metric_name, nq_data, nq_rf_data, experiment_labels, title, dataset_names):
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))
    axs = axs.flatten()

    # Define colors for each dataset pair
    colors = [
        ('#1f77b4', '#ff7f0e'),  # Dataset 1 colors (blue, orange)
        ('#2ca02c', '#d62728'),  # Dataset 2 colors (green, red)
        ('#9467bd', '#8c564b'),  # Dataset 3 colors (purple, brown)
        ('#e377c2', '#7f7f7f'),  # Dataset 4 colors (pink, gray)
    ]

    for i, ax in enumerate(axs):
        bar_width = 0.35
        x = np.arange(len(experiment_labels))
        
        # Plot bars for FT_NQ and FT_NQ_RF for the dataset
        bar1 = ax.bar(x - bar_width/2, nq_data[i], bar_width, label='FT with bm25', color=colors[i][0])
        bar2 = ax.bar(x + bar_width/2, nq_rf_data[i], bar_width, label='FT with splade', color=colors[i][1])
        
        # Add values on top of bars
        for rect in bar1:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)
        
        for rect in bar2:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

        # increase y_max to fit vertical numerical values
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        new_y_max = y_max + 0.4 * y_range
        ax.set_ylim([y_min, new_y_max])

        
        # Configure plot for the dataset
        ax.set_title(dataset_names[i])
        ax.set_xticks(x)
        ax.set_xticklabels(experiment_labels, rotation=45, ha="right")
        ax.set_ylabel(metric_name)
        ax.legend()

    fig.suptitle(title)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
    plt.savefig(f"tmp/figs/{title.replace(' ', '_').replace(':', '_')}.png")
    plt.show()

# Collect data for both NQ and NQ with Reformulated labels per dataset
def collect_experiment_data_per_dataset(experiments, dataset_dirs):
    matches, recalls, llmevals = [], [], []
    for dataset_dir in dataset_dirs:
        dataset_matches, dataset_recalls, dataset_llmevals = [], [], []
        for experiment in experiments:
            metrics = load_metrics(experiment, dataset_dir)
            dataset_matches.append(metrics['Match'])
            dataset_recalls.append(metrics['Recall'])
            dataset_llmevals.append(metrics['LLMeval'])
        matches.append(dataset_matches)
        recalls.append(dataset_recalls)
        llmevals.append(dataset_llmevals)
    return matches, recalls, llmevals

# Collect data for FT_NQ and FT_NQ_RF experiments
FT_bm25_matches, FT_bm25_recalls, FT_bm25_llmevals = collect_experiment_data_per_dataset(FT_bm25, dataset_dirs)
FT_splade_matches, FT_splade_recalls, FT_splade_llmevals = collect_experiment_data_per_dataset(FT_splade, dataset_dirs)

# Define experiment labels (using only the name part after 'll3_8b_')
experiment_labels = ["1 epoch MQA eval 0 shot", "1 epoch MQA eval BM25", "1 epoch MQA eval splade"] +\
      ["2 epochs MQA eval 0 shot", "2 epochs MQA eval BM25", "2 epochs MQA eval splade"]

# Plot the results using bar plots with subplots for 4 datasets
plot_metrics_bar_subplots('Match', FT_bm25_matches, FT_splade_matches, experiment_labels, 'Match Performance: FT_with_bm25 vs FT_with_splade', dataset_names)
plot_metrics_bar_subplots('Recall', FT_bm25_recalls, FT_splade_recalls, experiment_labels, 'Recall Performance: FT_with_bm25 vs FT_with_splade', dataset_names)
plot_metrics_bar_subplots('LLMeval', FT_bm25_llmevals, FT_splade_llmevals, experiment_labels, 'LLMeval Performance: FT_with_bm25 vs FT_with_splade', dataset_names)


## At test-time: 0-shot vs BM25 vs Splade

In [ ]:
zero_shot_experiments = [
    'll3_8b_0shot',
    'll3_8b_FT_NQ_1epoch_0shot',
    'll3_8b_FT_NQ_2epochs_0shot',
    'll3_8b_FT_NQ_RF_short_1epoch_0shot',
    'll3_8b_FT_NQ_RF_short_2epochs_0shot',
    'll3_8b_FT_MQA_BM25_5K_0shot',
    'll3_8b_FT_MQA_splade_5K_0shot',
    'll3_8b_FT_MQA_BM25_11K_0shot',
    'll3_8b_FT_MQA_splade_11K_0shot',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_06_0shot',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_09_0shot',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_06_0shot',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_09_0shot',
]

bm25_retrieval_experiments = [
    'll3_8b_bm25',
    'll3_8b_FT_NQ_1epoch_bm25',
    'll3_8b_FT_NQ_2epochs_bm25',
    'll3_8b_FT_NQ_RF_short_1epoch_bm25',
    'll3_8b_FT_NQ_RF_short_2epochs_bm25',
    'll3_8b_FT_MQA_BM25_5K_bm25',
    'll3_8b_FT_MQA_splade_5K_bm25',
    'll3_8b_FT_MQA_BM25_11K_bm25',
    'll3_8b_FT_MQA_splade_11K_bm25',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_06_bm25',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_09_bm25',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_06_bm25',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_09_bm25',
]

splade_retrieval_experiments = [
    'll3_8b_splade',
    'll3_8b_FT_NQ_1epoch_splade',
    'll3_8b_FT_NQ_2epochs_splade',
    'll3_8b_FT_NQ_RF_short_1epoch_splade',
    'll3_8b_FT_NQ_RF_short_2epochs_splade',
    'll3_8b_FT_MQA_BM25_5K_splade',
    'll3_8b_FT_MQA_splade_5K_splade',
    'll3_8b_FT_MQA_BM25_11K_splade',
    'll3_8b_FT_MQA_splade_11K_splade',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_06_splade',
    'll3_8b_FT_MQA_5K_distractors_3_ret_5_P_09_splade',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_06_splade',
    'll3_8b_FT_MQA_11K_distractors_3_ret_5_P_09_splade',
]


# Dataset names
dataset_names = ['ASQA', 'HotpotQA', 'BioASQ', 'SyllabusQA']

# Load metrics function
def load_metrics(subdir, dataset_dir):
    metrics_data = {'Match': 0.0, 'Recall': 0.0, 'LLMeval': 0.0}
    json_file_path = os.path.join(dataset_dir, subdir, 'eval_dev_metrics.json')
    if os.path.exists(json_file_path):
        with open(json_file_path, 'r') as json_file:
            metrics = json.load(json_file)
            metrics_data['Match'] = float(metrics.get('M', 0.0))
            metrics_data['Recall'] = float(metrics.get('Recall', 0.0))
            metrics_data['LLMeval'] = float(metrics.get('LLMeval_llama3.1:70b', 0.0))
    return metrics_data

# Function for collecting data
def collect_experiment_data_per_dataset(experiments, dataset_dirs):
    matches, recalls, llmevals = [], [], []
    for dataset_dir in dataset_dirs:
        dataset_matches, dataset_recalls, dataset_llmevals = [], [], []
        for experiment in experiments:
            metrics = load_metrics(experiment, dataset_dir)
            dataset_matches.append(metrics['Match'])
            dataset_recalls.append(metrics['Recall'])
            dataset_llmevals.append(metrics['LLMeval'])
        matches.append(dataset_matches)
        recalls.append(dataset_recalls)
        llmevals.append(dataset_llmevals)
    return matches, recalls, llmevals

# Collect data for 0-shot, BM25, and SPLADE experiments
zero_shot_matches, zero_shot_recalls, zero_shot_llmevals = collect_experiment_data_per_dataset(zero_shot_experiments, dataset_dirs)
bm25_matches, bm25_recalls, bm25_llmevals = collect_experiment_data_per_dataset(bm25_retrieval_experiments, dataset_dirs)
splade_matches, splade_recalls, splade_llmevals = collect_experiment_data_per_dataset(splade_retrieval_experiments, dataset_dirs)

# Plotting function for bar plots with subplots
def plot_metrics_bar_subplots(metric_name, zero_shot_data, bm25_data, splade_data, experiment_labels, title, dataset_names):
    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    axs = axs.flatten()

    # Define colors for each dataset triplet
    colors = [
        ('#1f77b4', '#ff7f0e', '#2ca02c'),  # Dataset 1 colors (blue, orange, green)
        ('#d62728', '#9467bd', '#8c564b'),  # Dataset 2 colors (red, purple, brown)
        ('#e377c2', '#7f7f7f', '#bcbd22'),  # Dataset 3 colors (pink, gray, yellow)
        ('#17becf', '#f7b6d2', '#aec7e8'),  # Dataset 4 colors (cyan, light pink, light blue)
    ]

    for i, ax in enumerate(axs):
        bar_width = 0.25
        x = np.arange(len(experiment_labels))
        
        # Plot bars for 0-shot, BM25, and SPLADE for the dataset
        bar1 = ax.bar(x - bar_width, zero_shot_data[i], bar_width, label='0-shot', color=colors[i][0])
        bar2 = ax.bar(x, bm25_data[i], bar_width, label='BM25', color=colors[i][1])
        bar3 = ax.bar(x + bar_width, splade_data[i], bar_width, label='SPLADE', color=colors[i][2])
        
        # Add values on top of bars
        for rect in bar1:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)
        
        for rect in bar2:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

        for rect in bar3:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

        # Adjust y-axis limits to fit numerical values
        y_min, y_max = ax.get_ylim()
        y_range = y_max - y_min
        new_y_max = y_max + 0.4 * y_range
        ax.set_ylim([y_min, new_y_max])

        
        # Configure plot for the dataset
        ax.set_title(dataset_names[i])
        ax.set_xticks(x)
        ax.set_xticklabels(experiment_labels, rotation=45, ha="right")
        ax.set_ylabel(metric_name)
        ax.legend()

    fig.suptitle(title)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
    plt.savefig(f"tmp/figs/{title.replace(' ', '_').replace(':', '_')}.png")
    plt.show()


# Define experiment labels (using only the name part after 'll3_8b_')
experiment_labels = [exp[7:-7] for exp in splade_retrieval_experiments]

# Plot the results using bar plots with subplots for 4 datasets
plot_metrics_bar_subplots('Match', zero_shot_matches, bm25_matches, splade_matches, experiment_labels, 'Match Performance: retrieve_bm25 vs retrieve_splade', dataset_names)
plot_metrics_bar_subplots('Recall', zero_shot_recalls, bm25_recalls, splade_recalls, experiment_labels, 'Recall Performance: retrieve_bm25 vs retrieve_splade', dataset_names)
plot_metrics_bar_subplots('LLMeval', zero_shot_llmevals, bm25_llmevals, splade_llmevals, experiment_labels, 'LLMeval Performance: retrieve_bm25 vs retrieve_splade', dataset_names)

# Distillation

In [ ]:
benchmarks_dir = "experiments/benchmarks_qwen"
distill_dir = "experiments/benchmarks_qwen_FT_MultiQA_distilled_mistral7b"
FT_dir = "experiments/benchmarks_qwen_FT_MultiQA_rf_short"

# load all runs in both directories
benchmarks_runs = [f for f in os.listdir(benchmarks_dir) if os.path.isdir(os.path.join(benchmarks_dir, f)) and 'ret' not in f]
distill_runs = [f for f in os.listdir(distill_dir) if os.path.isdir(os.path.join(distill_dir, f))]
ft_runs = [f for f in os.listdir(FT_dir) if os.path.isdir(os.path.join(FT_dir, f))]

def collect_experiment_data(dir, experiments=None):
    matchs, recalls, llmevals = dict(), dict(), dict()
    if experiments is None:
        for d in dir:
            with open(os.path.join(d, 'eval_dev_metrics.json'), 'r') as json_file:
                metrics = json.load(json_file)
            dataset_name = d.split('/')[1]
            matchs[dataset_name] = metrics['M']
            recalls[dataset_name] = metrics['Recall']
            llmevals[dataset_name] = float(metrics['LLMeval_llama3.1:70b']) if 'LLMeval_llama3.1:70b' in metrics else 0.0
    else:
        for experiment in experiments:
            if experiment.startswith('robustqa'):
                dataset_name = 'robustqa_' + experiment.split('_')[1]
            else:
                dataset_name = experiment.split('_')[0]
            with open(os.path.join(dir, experiment, 'eval_dev_metrics.json'), 'r') as json_file:
                metrics = json.load(json_file)
            matchs[dataset_name] = metrics['M']
            recalls[dataset_name] = metrics['Recall']
            llmevals[dataset_name] = float(metrics['LLMeval_llama3.1:70b']) if 'LLMeval_llama3.1:70b' in metrics else 0.0
    return matchs, recalls, llmevals

# Collect data for benchmarks and distill experiments
benchmarks_matches, benchmarks_recalls, benchmarks_llmevals = collect_experiment_data(benchmarks_dir, benchmarks_runs)
distill_matches, distill_recalls, distill_llmevals = collect_experiment_data(distill_dir, distill_runs)
ft_matches, ft_recalls, ft_llmevals = collect_experiment_data(FT_dir, ft_runs)

def plot_metrics_bar_subplots(metric_name, benchmarks_data, benchmarks_FT_data, distill_data, title):

    # sort data alphabetically by dataset name
    benchmarks_data = dict(sorted(benchmarks_data.items()))
    benchmarks_FT_data = dict(sorted(benchmarks_FT_data.items()))
    distill_data = dict(sorted(distill_data.items()))
    
    fig, ax = plt.subplots(figsize=(14, 10))

    bar_width = 0.2
    x = np.arange(len(benchmarks_data.keys()))

    # Plot bars for benchmarks and distill for the dataset
    bar0 = ax.bar(x - bar_width, benchmarks_data.values(), bar_width, label='Benchmarks (no FT)', color='b')
    bar1 = ax.bar(x, benchmarks_FT_data.values(), bar_width, label='Benchmarks (FT)', color='g')
    bar2 = ax.bar(x + bar_width, distill_data.values(), bar_width, label='Distill', color='r')

    for bar in [bar0, bar1, bar2]:
        for rect in bar:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

    # Configure plot for the dataset
    ax.set_xticks(x)
    ax.set_xticklabels(benchmarks_data.keys(), rotation=45, ha="right")
    ax.set_ylabel(metric_name)
    ax.legend()

    fig.suptitle(title)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
    plt.savefig(f"tmp/figs/{title.replace(' ', '_').replace(':', '_')}.png")
    plt.show()

# Plot the results using bar plots with subplots for 4 datasets
plot_metrics_bar_subplots('Match', benchmarks_matches, ft_matches, distill_matches, 'Qwen2.5-3b Match Performance: Base vs FT-rf-labels vs Distilled labels_mistral-7b')
plot_metrics_bar_subplots('Recall', benchmarks_recalls, ft_recalls, distill_recalls, 'Qwen2.5-3b Recall Performance: Base vs FT-rf-labels vs Distilled labels_mistral-7b')
plot_metrics_bar_subplots('LLMeval', benchmarks_llmevals, ft_llmevals, distill_llmevals, 'Qwen2.5-3b LLMeval Performance: Base vs FT-rf-labels vs Distilled labels_mistral-7b')

In [ ]:
distill_dir = "experiments/benchmarks_llama_FT_MultiQA_distilled_mistral7b"
distill_runs = [f for f in os.listdir(distill_dir) if os.path.isdir(os.path.join(distill_dir, f))]
standard_FT_runs = [
    "experiments/bioasq12b/ll38binstruct_splade_deberta_bioasqprompt_epochs/ll38binstruct_splade_deberta_bioasqprompt_2epoch",
    "experiments/covidqa/ll38binstruct_splade_deberta_covidqaprompt_epochs/ll38binstruct_splade_deberta_covidqaprompt_2epoch",
    "experiments/fiqa/ll38binstruct_splade_deberta_fiqaprompt_epochs/ll38binstruct_splade_deberta_fiqaprompt_2epoch",
    "experiments/paraphraserc/ll38binstruct_splade_deberta_paraphrasercprompt_epochs/ll38binstruct_splade_deberta_paraphrasercprompt_2epoch",
    "experiments/robustqa_lifestyle/ll38binstruct_splade_deberta_lifestyleprompt_epochs/ll38binstruct_splade_deberta_techqaprompt_2epoch",
    "experiments/robustqa_recreation/ll38binstruct_splade_deberta_recreationprompt_epochs/ll38binstruct_splade_deberta_recreationprompt_2epoch",
    "experiments/robustqa_science/ll38binstruct_splade_deberta_scienceprompt_epochs/ll38binstruct_splade_deberta_scienceprompt_2epoch",
    "experiments/robustqa_technology/ll38binstruct_splade_deberta_technologyprompt_epochs/ll38binstruct_splade_deberta_technologyprompt_2epoch",
    "experiments/robustqa_writing/ll38binstruct_splade_deberta_writingprompt_epochs/ll38binstruct_splade_deberta_writingprompt_2epoch",
    "experiments/searchqa/ll38binstruct_splade_deberta_searchqaprompt_epochs/ll38binstruct_splade_deberta_searchqaprompt_2epoch",
    "experiments/syllabusQA/ll38binstruct_splade_deberta_syllabusqaprompt_epochs/ll38binstruct_splade_deberta_syllabusqaprompt_2epoch",
    "experiments/techqa/ll38binstruct_splade_deberta_techqaprompt_epochs/ll38binstruct_splade_deberta_techqaprompt_2epoch",
]
benchmarks_llama_runs = [
    "experiments/bioasq12b/ll3_8b_splade_deberta_bioasqprompt_2",
    "experiments/covidqa/llama3_8b_splade_deberta_covidqaprompt",
    "experiments/fiqa/llama3_8b_splade_deberta_advprompt",
    "experiments/paraphraserc/llama3_8b_splade_deberta_advprompt",
    "experiments/robustqa_lifestyle/llama3_8b_splade_deberta_lifestyleprompt",
    "experiments/robustqa_recreation/llama3_8b_splade_deberta_recreationprompt",
    "experiments/robustqa_science/llama3_8b_splade_deberta_scienceprompt",
    "experiments/robustqa_technology/llama3_8b_splade_deberta_techprompt",
    "experiments/robustqa_writing/llama3_8b_splade_deberta_writingprompt",
    "experiments/searchqa/llama3_8b_splade_deberta_techprompt",
    "experiments/syllabusQA/llama3_8b_splade_deberta_syllqaprompt",
    "experiments/techqa/llama3_8b_splade_deberta_techprompt",
]

def collect_experiment_data(dir, experiments=None):
    matchs, recalls, llmevals = dict(), dict(), dict()
    if experiments is None:
        for d in dir:
            with open(os.path.join(d, 'eval_dev_metrics.json'), 'r') as json_file:
                metrics = json.load(json_file)
            dataset_name = d.split('/')[1]
            matchs[dataset_name] = metrics['M']
            recalls[dataset_name] = metrics['Recall']
            llmevals[dataset_name] = float(metrics['LLMeval_llama3.1:70b'])
    else:
        for experiment in experiments:
            if experiment.startswith('robustqa'):
                dataset_name = 'robustqa_' + experiment.split('_')[1]
            else:
                dataset_name = experiment.split('_')[0]
            with open(os.path.join(dir, experiment, 'eval_dev_metrics.json'), 'r') as json_file:
                metrics = json.load(json_file)
            matchs[dataset_name] = metrics['M']
            recalls[dataset_name] = metrics['Recall']
            llmevals[dataset_name] = float(metrics['LLMeval_llama3.1:70b'])
    return matchs, recalls, llmevals

# Collect data for benchmarks and distill experiments
benchmarks_matches, benchmarks_recalls, benchmarks_llmevals = collect_experiment_data(benchmarks_llama_runs)
benchmarks_FT_matches, benchmarks_FT_recalls, benchmarks_FT_llmevals = collect_experiment_data(standard_FT_runs)
distill_matches, distill_recalls, distill_llmevals = collect_experiment_data(distill_dir, distill_runs)

# plot a recall and llmeval bar plot pairing benchmarks and distill next to each other for each dataset

def plot_metrics_bar_subplots(metric_name, benchmarks_data, benchmarks_FT_data, distill_data, title):

    # sort data alphabetically by dataset name
    benchmarks_data = dict(sorted(benchmarks_data.items()))
    benchmarks_FT_data = dict(sorted(benchmarks_FT_data.items()))
    distill_data = dict(sorted(distill_data.items()))
    
    fig, ax = plt.subplots(figsize=(14, 10))

    bar_width = 0.2
    x = np.arange(len(benchmarks_data.keys()))

    # Plot bars for benchmarks and distill for the dataset
    bar0 = ax.bar(x - bar_width, benchmarks_data.values(), bar_width, label='Benchmarks (no FT)', color='b')
    bar1 = ax.bar(x, benchmarks_FT_data.values(), bar_width, label='Benchmarks (FT)', color='g')
    bar2 = ax.bar(x + bar_width, distill_data.values(), bar_width, label='Distill', color='r')

    for bar in [bar0, bar1, bar2]:
        for rect in bar:
            height = rect.get_height()
            ax.text(rect.get_x() + rect.get_width() / 2.0, height + 0.02, f'{height:.2f}', ha='center', va='bottom', rotation=90)

    # Configure plot for the dataset
    ax.set_xticks(x)
    ax.set_xticklabels(benchmarks_data.keys(), rotation=45, ha="right")
    ax.set_ylabel(metric_name)
    ax.legend()

    fig.suptitle(title)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
    plt.savefig(f"tmp/figs/{title.replace(' ', '_').replace(':', '_')}.png")
    plt.show()

# Plot the results using bar plots with subplots for 4 datasets
plot_metrics_bar_subplots('Match', benchmarks_matches, benchmarks_FT_matches, distill_matches, 'Llama3-8b Match Performance: Base vs FT-rf-labels vs Distilled labels_mistral-7b')
plot_metrics_bar_subplots('Recall', benchmarks_recalls, benchmarks_FT_recalls, distill_recalls, 'Llama3-8b Recall Performance — Base vs FT-rf-labels vs Distilled labels_mistral-7b')
plot_metrics_bar_subplots('LLMeval', benchmarks_llmevals, benchmarks_FT_llmevals, distill_llmevals, 'Llama3-8b LLMeval Performance — Base vs FT-rf-labels vs Distilled labels_mistral-7b')

# Plot SOLAR-10.7B RAGChecker metrics on 6 benchmarks

In [ ]:
def generate_spider_diagram(metrics_dict, filename=None):
    """
    Generate and overlap multiple spider diagrams for multiple datasets.
    
    Args:
        metrics_dict (dict): Dictionary where keys are dataset names and values are metrics.
        filename (str, optional): Path to save the resulting spider diagram.
    """
    categories = list(next(iter(metrics_dict.values())).keys())
    N = len(categories)

    # Angle for each axis in the spider plot
    angles = [n / float(N) * 2 * math.pi for n in range(N)]
    angles += angles[:1]  # close the loop

    # Set up the figure
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
    plt.xticks(angles[:-1], categories, color='black', size=15)
    ax.tick_params(axis='x', pad=30)
    ax.set_rlabel_position(0)
    plt.yticks([0.2, 0.4, 0.6, 0.8], ["0.2", "0.4", "0.6", "0.8"], color="grey", size=10)
    plt.ylim(0, 1)

    # Define a color palette
    colors = plt.cm.Paired(np.linspace(0, 1, len(metrics_dict)))

    # Plot each dataset's metrics
    for (dataset, metrics), color in zip(metrics_dict.items(), colors):
        values = list(metrics.values())
        values += values[:1]  # close the loop
        ax.plot(angles, values, linewidth=1.5, linestyle='solid', label=dataset, color=color)
        ax.fill(angles, values, color=color, alpha=0.1)

    # Add legend
    ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.0), fontsize=15)

    # Save or show the plot
    if filename:
        plt.title("RAGChecker Metrics for SOLAR-10.7B (spladeberta document chunks), across datasets", size=18)
        plt.savefig(filename, bbox_inches='tight')
        plt.clf()
    else:
        plt.show()

root = "experiments/runs_hackathon/solar"
all_metrics = {}

def parse(met):
    out = dict()
    # add overall_metrics keys
    for key, value in met["overall_metrics"].items():
        out[key.replace("_", " ") + r"$\uparrow$"] = value
    # add generator_metrics keys
    for key, value in met["generator_metrics"].items():
        if key not in ["relevant_noise_sensitivity", "irrelevant_noise_sensitivity"]:
            out[key.replace("_", " ") + r"$\uparrow$"] = value
        else:
            out[key.replace("_", " ") + r"$\downarrow$"] = value
    # add retriever metrics
    for key, value in met["retriever_metrics"].items():
        out[key.replace("_", " ") + r"$\uparrow$"] = value
    return out



In [ ]:

for subdir in os.listdir(root):
    if subdir.endswith("solar107b"):
        dataset = subdir.split("/")[-1].split("_")[0]
        metrics_file = f"{root}/{subdir}/ragchecker_dev_metrics_qwen.json"
        with open(metrics_file, "r") as f:
            metrics = json.load(f)
            all_metrics[dataset] = parse(metrics)

# Generate and show/save the spider diagram for all datasets
output_file = "tmp/figs/ragchecker_benchmarks.png"
generate_spider_diagram(all_metrics, filename=output_file)

In [ ]:
root = "experiments/runs_hackathon/solar_generalprompt"
all_metrics = {}

for subdir in os.listdir(root):
    if subdir.endswith("solar107b"):
        dataset = subdir.split("/")[-1].split("_")[0]
        metrics_file = f"{root}/{subdir}/ragchecker_dev_metrics_qwen.json"
        with open(metrics_file, "r") as f:
            metrics = json.load(f)
            all_metrics[dataset] = parse(metrics)

# Generate and show/save the spider diagram for all datasets
output_file = "tmp/figs/ragchecker_benchmarks_selfknwlgprompt.png"
generate_spider_diagram(all_metrics, filename=output_file)

In [ ]:
base_model_path = "experiments/tune_diff_attn_learnlambda/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_base_init_alllayers/train/"

def lambda_init_fn(depth):
    return 0.8 - 0.6 * math.exp(-0.3 * depth)

def lambda_fn(lambda_q1, lambda_k1, lambda_q2, lambda_k2, lambda_init):
    lambda_1 = torch.exp(torch.sum(lambda_q1 * lambda_k1, dim=-1).float())
    lambda_2 = torch.exp(torch.sum(lambda_q2 * lambda_k2, dim=-1).float())
    lambda_full = lambda_1 - lambda_2 + lambda_init
    return lambda_full

lambdas_per_layer_per_chkpnt = defaultdict(list)
# first add the initial lambda values (checkpoint 0)
for layer in range(32):
    lambda_q1 = torch.zeros(128, dtype=torch.float32).normal_(mean=0,std=0.1)
    lambda_k1 = torch.zeros(128, dtype=torch.float32).normal_(mean=0,std=0.1)
    lambda_q2 = torch.zeros(128, dtype=torch.float32).normal_(mean=0,std=0.1)
    lambda_k2 = torch.zeros(128, dtype=torch.float32).normal_(mean=0,std=0.1)
    lambda_init = lambda_init_fn(layer)
    lambda_full = lambda_fn(lambda_q1, lambda_k1, lambda_q2, lambda_k2, lambda_init)
    lambdas_per_layer_per_chkpnt[layer].append(lambda_full)
# iterate over checkpoint folders
folders = sorted([os.path.join(base_model_path, e) for e in os.listdir(base_model_path)], key=lambda x: int(x.split('/')[-1].split('-')[-1]))
for model_path in folders:
    state_dict = safetensors.torch.load_file(os.path.join(model_path, "model.safetensors"))
    state_dict.keys()
    for layer in range(32):
        lambda_init = lambda_init_fn(layer)
        lambda_q1 = state_dict[f'model.layers.{layer}.self_attn.lambda_q1']
        lambda_k1 = state_dict[f'model.layers.{layer}.self_attn.lambda_k1']
        lambda_q2 = state_dict[f'model.layers.{layer}.self_attn.lambda_q2']
        lambda_k2 = state_dict[f'model.layers.{layer}.self_attn.lambda_k2']
        lambda_full = lambda_fn(lambda_q1, lambda_k1, lambda_q2, lambda_k2, lambda_init)
        lambdas_per_layer_per_chkpnt[layer].append(lambda_full)

In [ ]:
# plot the lambda values for each layer in each checkpoint
# x axis is the checkpoint value (sorted by checkpoint name)
# y axis is lambda value, one line for each layer
# plot a line for each layer
# x axis is the checkpoint number

fig, ax = plt.subplots(figsize=(10, 8))
x = np.arange(len(folders)+1)
for layer in range(32):
    y = np.array([lambdas_per_layer_per_chkpnt[layer][0]] + [lambdas_per_layer_per_chkpnt[layer][i+1].item() for i in range(len(folders))])
    ax.plot(x, y, label=f'layer {layer}')
    std_dev = np.array([0.161] + [0 for _ in range(len(folders))])
    ax.fill_between(x, y - std_dev, y + std_dev, alpha=0.2)
    # write on the line the layer number
    ax.text(x[-1], lambdas_per_layer_per_chkpnt[layer][-1].item(), f'layer {layer}', ha='left', va='center')
ax.set_xticks(x)
ax.set_xticklabels(["init"]+[e.split('/')[-1] for e in folders], rotation=45, ha="right")
plt.title(f"$\lambda$ values per layer for model {base_model_path.split('/')[-3]}")
plt.show()

In [ ]:
layer = 0
lamda_fulls = []
for i in range(100000):
    lambda_q1 = torch.zeros(128, dtype=torch.float32).normal_(mean=0,std=0.1)
    lambda_k1 = torch.zeros(128, dtype=torch.float32).normal_(mean=0,std=0.1)
    lambda_q2 = torch.zeros(128, dtype=torch.float32).normal_(mean=0,std=0.1)
    lambda_k2 = torch.zeros(128, dtype=torch.float32).normal_(mean=0,std=0.1)
    lambda_init = lambda_init_fn(layer)
    lambda_full = lambda_fn(lambda_q1, lambda_k1, lambda_q2, lambda_k2, lambda_init)
    lamda_fulls.append(lambda_full)

plt.hist(lamda_fulls, bins=100)
plt.show()

In [ ]:
import os
import math
import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict
import safetensors.torch

# Define functions for lambda calculations
def lambda_init_fn(depth):
    return 0.8 - 0.6 * math.exp(-0.3 * depth)

def lambda_fn(lambda_q1, lambda_k1, lambda_q2, lambda_k2, lambda_init):
    lambda_1 = torch.exp(torch.sum(lambda_q1 * lambda_k1, dim=-1).float())
    lambda_2 = torch.exp(torch.sum(lambda_q2 * lambda_k2, dim=-1).float())
    lambda_full = lambda_1 - lambda_2 + lambda_init
    return lambda_full

# Define multiple base model paths
base_model_paths = [
    "experiments/tune_diff_attn_learnlambda/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_base_init_alllayers",
    "experiments/tune_diff_attn_learnlambda/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_base_init_alllayers_rightloraonly",
    "experiments/tune_diff_attn_learnlambda_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_base_init_alllayers",
    "experiments/tune_diff_attn_learnlambda_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_base_init_alllayers_rightloraonly",
]
base_model_paths = [os.path.join(e, 'train/') for e in base_model_paths]

# Process each base model path
global_min, global_max = float('inf'), float('-inf')
lambdas_data = {}

for base_model_path in base_model_paths:
    lambdas_per_layer_per_chkpnt = defaultdict(list)
    
    # Add initial lambda values (checkpoint 0)
    for layer in range(32):
        lambda_q1 = torch.zeros(128, dtype=torch.float32).normal_(mean=0, std=0.1)
        lambda_k1 = torch.zeros(128, dtype=torch.float32).normal_(mean=0, std=0.1)
        lambda_q2 = torch.zeros(128, dtype=torch.float32).normal_(mean=0, std=0.1)
        lambda_k2 = torch.zeros(128, dtype=torch.float32).normal_(mean=0, std=0.1)
        lambda_init = lambda_init_fn(layer)
        lambda_full = lambda_fn(lambda_q1, lambda_k1, lambda_q2, lambda_k2, lambda_init)
        lambdas_per_layer_per_chkpnt[layer].append(lambda_full)
        global_min = min(global_min, lambda_full.item())
        global_max = max(global_max, lambda_full.item())
    
    # Iterate over checkpoint folders
    folders = sorted(
        [os.path.join(base_model_path, e) for e in os.listdir(base_model_path) if os.path.isdir(os.path.join(base_model_path, e))],
        key=lambda x: int(x.split('/')[-1].split('-')[-1])
    )
    for model_path in folders:
        state_dict = safetensors.torch.load_file(os.path.join(model_path, "model.safetensors"))
        for layer in range(32):
            lambda_init = lambda_init_fn(layer)
            lambda_q1 = state_dict[f'model.layers.{layer}.self_attn.lambda_q1']
            lambda_k1 = state_dict[f'model.layers.{layer}.self_attn.lambda_k1']
            lambda_q2 = state_dict[f'model.layers.{layer}.self_attn.lambda_q2']
            lambda_k2 = state_dict[f'model.layers.{layer}.self_attn.lambda_k2']
            lambda_full = lambda_fn(lambda_q1, lambda_k1, lambda_q2, lambda_k2, lambda_init)
            lambdas_per_layer_per_chkpnt[layer].append(lambda_full)
            global_min = min(global_min, lambda_full.item())
            global_max = max(global_max, lambda_full.item())
    
    lambdas_data[base_model_path] = (lambdas_per_layer_per_chkpnt, folders)

# Plot results
fig, axes = plt.subplots(len(base_model_paths), 1, figsize=(12, 8 * len(base_model_paths)), sharey=True)

for i, base_model_path in enumerate(base_model_paths):
    ax = axes[i] if len(base_model_paths) > 1 else axes
    lambdas_per_layer_per_chkpnt, folders = lambdas_data[base_model_path]
    
    x = np.arange(len(folders) + 1)
    for layer in range(32):
        y = np.array([lambdas_per_layer_per_chkpnt[layer][0].item()] + 
                     [lambdas_per_layer_per_chkpnt[layer][j + 1].item() for j in range(len(folders))])
        ax.plot(x, y, label=f'Layer {layer}')
        std_dev = np.array([0.161] + [0 for _ in range(len(folders))])
        ax.fill_between(x, lambda_init_fn(layer) - std_dev, lambda_init_fn(layer) + std_dev, alpha=0.2)
        # write on the line the layer number
        ax.text(x[-1], lambdas_per_layer_per_chkpnt[layer][-1].item(), f'layer {layer}', ha='left', va='center')
        
    ax.set_xticks(x)
    ax.set_xticklabels(["init"] + [e.split('/')[-1] for e in folders], rotation=45, ha="right")
    ax.set_ylim(global_min, global_max)
    title = ""
    if "rightloraonly" in base_model_path:
        title += "Right Lora Only, "
    else:
        title += "Adapters on pos and neg terms, "
    if "nogroupnorm" in base_model_path:
        title += "No Group Norm "
    else:
        title += "With Group Norm"
    ax.set_title(f"$\lambda$ along training, {title}")
    ax.set_ylabel("$\lambda$ value")
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize='small')

plt.tight_layout()
plt.xlabel("Checkpoint")
plt.savefig("diff_attn_lambda_values_along_training_per_layer.png")
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]
dirs = """experiments/control_nq_llama/llama38binstruct_evaltop3 
experiments/control_nq_llama/train_Lora_NQ_llama38b_instruct_spladeberta_top3_basicprompt/eval_top3 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_r256/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_r256/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly_r512/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_rightloraonly_r512/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly_r512/eval""".split()

labels = ["llama", "llama-lora", "diff-attn-lambda=0.1", "diff-attn-lambda=0.001", "diff-attn-lambda=0.1-r256", "diff-attn-lambda=0.001-r256", 
          "diff-attn-lambda=0.1-rightlora", "diff-attn-lambda=0.001-rightlora", "diff-attn-lambda=0.1-rightlora-r512", "diff-attn-lambda=0.001-rightlora-r512", 
          "diff-attn-lambda=0.9 (with norm)", "diff-attn-lambda=0.9-rightlora (with norm)", "diff-attn-lambda=0.9-rightlora-r512 (with norm)"]

# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get("Recall", 0)
                recall_data[domain].append((labels[i], recall))
        else:
            recall_data[domain].append((labels[i], None))

# Plotting
plt.figure(figsize=(15, 10))
x_positions = np.arange(len(dirs))
bar_width = 0.8 / len(domains)  # Adjust bar width to fit all domains

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0
    
    plt.bar(x_positions + i * bar_width, recalls, bar_width, label=domain)

plt.xticks(x_positions + bar_width * (len(domains) / 2 - 0.5), dirs, rotation=90, fontsize=8)
plt.xlabel("Directories")
plt.ylabel("Recall")
plt.title("Recall Across Domains")
plt.legend(title="Domains", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(axis='y')

# Save the plot
# plt.savefig("figs/diff_attn_eval_domains_recall.png")
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]
dirs = """experiments/control_nq_llama/llama38binstruct_evaltop3 
experiments/control_nq_llama/train_Lora_NQ_llama38b_instruct_spladeberta_top3_basicprompt/eval_top3 
experiments/control_nq_llama/train_Lora_r512_NQ_llama38b_instruct_spladeberta_top3_basicprompt/eval_top3 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_r256/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_r256/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly_r512/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_rightloraonly_r512/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly_r512/eval""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.001", "diff-attn-lambda=0.1-r256", "diff-attn-lambda=0.001-r256", 
          "diff-attn-lambda=0.1-rightlora", "diff-attn-lambda=0.001-rightlora", "diff-attn-lambda=0.1-rightlora-r512", "diff-attn-lambda=0.001-rightlora-r512", 
          "diff-attn-lambda=0.9 (with norm)", "diff-attn-lambda=0.9-rightlora (with norm)", "diff-attn-lambda=0.9-rightlora-r512 (with norm)"]

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

metric = "EM"
metric = "M"
metric = "Recall"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            recall_data[domain].append((labels[i], None))

# Plotting
num_domains = len(domains)
fig, axes = plt.subplots(num_domains, 1, figsize=(8, 7 * num_domains))

for i, (ax, (domain, values)) in enumerate(zip(axes, recall_data.items())):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, recalls, color=colors[i])
    ax.set_title(domain + " " + metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_eval_domains_{metric}.png")
plt.show()

In [ ]:
dirs = """experiments/nih_hash/llama38binstruct 
experiments/nih_hash/llama-lora-NQ 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_r256/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_r256/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_rightloraonly/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly_r512/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_rightloraonly_r512/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_r256/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_rightloraonly/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_rightloraonly_r512/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly/eval/nih_hash 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly_r512/eval/nih_hash""".split()

labels = ["llama", "llama-lora", "diff-attn-lambda=0.1", "diff-attn-lambda=0.001", "diff-attn-lambda=0.1-r256", "diff-attn-lambda=0.001-r256", 
          "diff-attn-lambda=0.1-rightlora", "diff-attn-lambda=0.001-rightlora", "diff-attn-lambda=0.1-rightlora-r512", "diff-attn-lambda=0.001-rightlora-r512", 
          "diff-attn-lambda=0.5", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora", "diff-attn-lambda=0.5-rightlora-r512",
          "diff-attn-lambda=0.9 (with norm)", "diff-attn-lambda=0.9-rightlora (with norm)", "diff-attn-lambda=0.9-rightlora-r512 (with norm)"]

nih_hash_matches = dict()
magic_phrase_attn_score = dict()

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            nih_hash_matches[labels[i]] = metrics.get("M", 0)
            if "att" in metrics:
                magic_phrase_attn_score[labels[i]] = metrics["att"].get("att_last_magic", 0)
            elif "att_last_magic" in metrics:
                magic_phrase_attn_score[labels[i]] = metrics["att_last_magic"]
            else:
                print(f"Missing 'att' or 'att_last_magic' in {eval_file}")
                magic_phrase_attn_score[labels[i]] = "TBD"            
    else:
        print(f"Missing {eval_file}")
        nih_hash_matches[labels[i]] = "TBD"
        magic_phrase_attn_score[labels[i]] = "TBD"

# print ready for github markdown
print("| Checkpoint | NIH-hash match | Needle attn score |")
print("| --- | --- | --- |")
for label in labels:
    if nih_hash_matches[label] != "TBD" and magic_phrase_attn_score[label] != "TBD":
        print(f"| {label} | {nih_hash_matches[label]:.3f} | {magic_phrase_attn_score[label]:.6f} |")
    else:
        print(f"| {label} | {nih_hash_matches[label]} | {magic_phrase_attn_score[label]} |")

In [ ]:
dirs = """experiments/control_nq_llama/eval_NQ_llama3_8b_instruct_spladeberta_basicprompt 
experiments/control_nq_llama/train_Lora_NQ_llama38b_instruct_spladeberta_top3_basicprompt 
experiments/control_nq_llama/train_Lora_r512_NQ_llama38b_instruct_spladeberta_top3_basicprompt 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_r256 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_r256 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_rightloraonly 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly_r512 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda0001_base_init_alllayers_rightloraonly_r512 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_r256 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_rightloraonly 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_rightloraonly_r512 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda09_base_init_alllayers 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda09_base_init_alllayers_r256 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda09_base_init_alllayers_rightloraonly 
experiments/tune_diff_attn_nq_lambda_spladeberta_nogroupnorm/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda09_base_init_alllayers_rightloraonly_r512 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly 
experiments/tune_diff_attn_nq_lambda_spladeberta/train_LoraDiffAtt_NQ_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly_r512""".split()

labels = ["llama", "llama-lora", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.001", "diff-attn-lambda=0.1-r256", "diff-attn-lambda=0.001-r256", 
          "diff-attn-lambda=0.1-rightlora", "diff-attn-lambda=0.001-rightlora", "diff-attn-lambda=0.1-rightlora-r512", "diff-attn-lambda=0.001-rightlora-r512", 
          "diff-attn-lambda=0.5", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora", "diff-attn-lambda=0.5-rightlora-r512",
            "diff-attn-lambda=0.9", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora", "diff-attn-lambda=0.9-rightlora-r512",
          "diff-attn-lambda=0.9 (with norm)", "diff-attn-lambda=0.9-rightlora (with norm)", "diff-attn-lambda=0.9-rightlora-r512 (with norm)"]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [em, m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [em, m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [em, m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Eval NQ metric = "+metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_eval_NQ_metrics.png")
plt.show()


In [ ]:
from datasets import load_from_disk
ds = load_from_disk("datasets/MultiQA_train")

In [ ]:
ds

In [ ]:
# plot dataset label lengths
import matplotlib.pyplot as plt
import numpy as np

label_lengths = [len(e) for e in ds['label']]
plt.hist(label_lengths, bins=100)
plt.show()

In [ ]:
sorted(label_lengths)[-150:]

In [ ]:
dirs = """experiments/control_nq_llama/eval_MultiQA_llama3_8b_instruct_basicprompt 
experiments/control_nq_llama/train_Lora_MultiQA_llama38b_instruct_spladeberta_top3_basicprompt 
experiments/control_nq_llama/train_Lora_r512_MultiQA_llama38b_instruct_spladeberta_top3_basicprompt 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_r256 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly_r512 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_r256 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_rightloraonly 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_rightloraonly_r512 
experiments/tune_diff_attn_multiqa_highlambda_withnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers 
experiments/tune_diff_attn_multiqa_highlambda_withnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly_r256 
experiments/tune_diff_attn_multiqa_highlambda_withnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly 
experiments/tune_diff_attn_multiqa_highlambda_withnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly_r512 """.split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-r256",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r512", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r512",
            "diff-attn-lambda=0.9-r32", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r64", "diff-attn-lambda=0.9-rightlora-r512",
          "diff-attn-lambda=0.9-r32 (with norm)", "diff-attn-lambda=0.9-r256 (with norm)", "diff-attn-lambda=0.9-rightlora-r64 (with norm)", "diff-attn-lambda=0.9-rightlora-r512 (with norm)"]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [em, m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [em, m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [em, m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Eval MultiQA metric = "+metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_eval_MQA_metrics.png")
plt.show()


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]
dirs = """experiments/control_nq_llama/llama38binstruct_evaltop3 
experiments/control_nq_llama/train_Lora_MultiQA_llama38b_instruct_spladeberta_top3_basicprompt/eval_top3 
experiments/control_nq_llama/train_Lora_r512_MultiQA_llama38b_instruct_spladeberta_top3_basicprompt/eval_top3 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers/eval 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_r256/eval 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda01_base_init_alllayers_rightloraonly_r512/eval 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers/eval 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_r256/eval 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_multiqa_lowlambda_nogroupnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_nogroupnorm_lambda05_base_init_alllayers_rightloraonly_r512/eval 
experiments/tune_diff_attn_multiqa_highlambda_withnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers/eval 
experiments/tune_diff_attn_multiqa_highlambda_withnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly_r256/eval 
experiments/tune_diff_attn_multiqa_highlambda_withnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly/eval 
experiments/tune_diff_attn_multiqa_highlambda_withnorm/train_LoraDiffAtt_multiqa_llama38b_instruct_spladeberta_top3_basicprompt_lambda09_base_init_alllayers_rightloraonly_r512/eval """.split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-r256",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r512", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r512",
            "diff-attn-lambda=0.9-r32", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r64", "diff-attn-lambda=0.9-rightlora-r512",
          "diff-attn-lambda=0.9-r32 (with norm)", "diff-attn-lambda=0.9-r256 (with norm)", "diff-attn-lambda=0.9-rightlora-r64 (with norm)", "diff-attn-lambda=0.9-rightlora-r512 (with norm)"]

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

metric = "EM"
metric = "M"
metric = "Recall"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

# Plotting
num_domains = len(domains)
fig, axes = plt.subplots(num_domains, 1, figsize=(8, 7 * num_domains))

for i, (ax, (domain, values)) in enumerate(zip(axes, recall_data.items())):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, recalls, color=colors[i])
    ax.set_title(domain + " " + metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_MultiQA_eval_domains_{metric}.png")
plt.show()

In [ ]:
dirs = """experiments/llama32_1B_instruct/baseline_multiqa 
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt 
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.8_r32 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.8_r256 
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.8_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.8_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.999_r32
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.999_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.999_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.999_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.1_r32
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.1_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.1_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.1_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.2_r32
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.2_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.2_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.2_r512
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-r256",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r512", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r512",
          "diff-attn-lambda=0.8-r32", "diff-attn-lambda=0.8-r256", "diff-attn-lambda=0.8-rightlora-r64", "diff-attn-lambda=0.8-rightlora-r512",
          "diff-attn-lambda=0.9-r32", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r64", "diff-attn-lambda=0.9-rightlora-r512",
            "diff-attn-lambda=0.999-r32", "diff-attn-lambda=0.999-r256", "diff-attn-lambda=0.999-rightlora-r64", "diff-attn-lambda=0.999-rightlora-r512",
            "diff-attn-lambda=1.1-r32", "diff-attn-lambda=1.1-r256", "diff-attn-lambda=1.1-rightlora-r64", "diff-attn-lambda=1.1-rightlora-r512",
            "diff-attn-lambda=1.2-r32", "diff-attn-lambda=1.2-r256", "diff-attn-lambda=1.2-rightlora-r64", "diff-attn-lambda=1.2-rightlora-r512",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [em, m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [em, m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [em, m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Eval MultiQA metric = "+metric+ " (lambda > 0.5 with groupnorm)", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_MQA_metrics.png")
plt.show()


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]
dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.8_r32/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.8_r256/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.8_r64/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.8_r512/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r256/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.999_r32/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.999_r256/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.999_r64/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.999_r512/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.1_r32/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.1_r256/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.1_r64/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.1_r512/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.2_r32/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.2_r256/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.2_r64/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_1.2_r512/eval
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-r256",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r512", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r512",
          "diff-attn-lambda=0.8-r32", "diff-attn-lambda=0.8-r256", "diff-attn-lambda=0.8-rightlora-r64", "diff-attn-lambda=0.8-rightlora-r512",
          "diff-attn-lambda=0.9-r32", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r64", "diff-attn-lambda=0.9-rightlora-r512",
            "diff-attn-lambda=0.999-r32", "diff-attn-lambda=0.999-r256", "diff-attn-lambda=0.999-rightlora-r64", "diff-attn-lambda=0.999-rightlora-r512",
            "diff-attn-lambda=1.1-r32", "diff-attn-lambda=1.1-r256", "diff-attn-lambda=1.1-rightlora-r64", "diff-attn-lambda=1.1-rightlora-r512",
            "diff-attn-lambda=1.2-r32", "diff-attn-lambda=1.2-r256", "diff-attn-lambda=1.2-rightlora-r64", "diff-attn-lambda=1.2-rightlora-r512",
          ]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

metric = "EM"
# metric = "M"
# metric = "Recall"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

# Plotting
num_domains = len(domains)
fig, axes = plt.subplots(num_domains, 1, figsize=(8, 7 * num_domains))

for i, (ax, (domain, values)) in enumerate(zip(axes, recall_data.items())):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, recalls, color=colors[i])
    ax.set_title(domain + " " + metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_MultiQA_eval_domains_{metric}.png")
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "nih_long_context", "nq"
]
dirs = """experiments/llama32_1B_instruct/baseline_nih_long_context
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/nih_long_context
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r256/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512/eval/nih_long_context
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-r256",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r512", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r512",
          "diff-attn-lambda=0.9-r32", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r64", "diff-attn-lambda=0.9-rightlora-r512",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Long-context needle-in-haystack metric = "+metric+ " (lambda > 0.5 with groupnorm)", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_NIH.png")
plt.show()


In [ ]:
'''
code for plot taken from https://github.com/gkamradt/LLMTest_NeedleInAHaystack/blob/main/viz/CreateVizFromLLMTesting.ipynb
'''

import os
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import glob


# Directories and domains
domains = [
    "nih_long_context", "nq"
]
dirs = """experiments/llama32_1B_instruct/baseline_nih_long_context
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/nih_long_context
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r256/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512/eval/nih_long_context
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-r256",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r512", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r512",
          "diff-attn-lambda=0.9-r32", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r64", "diff-attn-lambda=0.9-rightlora-r512",
          ]


VIEW = "Document Depth"
# VIEW = "Context Length"
em = "EM"
m = "M"
recall = "Recall"
metric = recall
data = []

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_out.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            output = json.load(f)
            for row in output:
                # q_id = mnih_long_context_cl100_dp0
                context_length = int(row["q_id"].split("_")[3][2:])
                document_depth = int(row["q_id"].split("_")[4][2:])
                score = row[metric]
                data.append({
                    "Document Depth": document_depth,
                    "Context Length": context_length,
                    "Score": score,
                    "dir": labels[i],
                })
    else:
        print(f"Missing {eval_file}")

# plot a heatmap with y axis context length and x axis each directory (average across all document depths)
# Create a DataFrame
cmap = LinearSegmentedColormap.from_list("custom_cmap", ["#F0496E", "#EBB839", "#0CD79F"])

df = pd.DataFrame(data)
pivot_table = pd.pivot_table(df, values='Score', index=[VIEW, 'dir'], aggfunc='mean').reset_index() # This will aggregate
pivot_table = pivot_table.pivot(index=VIEW, columns="dir", values="Score") # This will turn into a proper pivot
# reorder colums in same order as labels
pivot_table = pivot_table[labels]

# Create the heatmap with better aesthetics
plt.figure(figsize=(17.5, 8))  # Can adjust these dimensions as needed
sns.heatmap(
    pivot_table,
    # annot=True,
    fmt="g",
    cmap=cmap,
    cbar_kws={'label': 'Recall'}
)

# More aesthetics
plt.title('Long-context needle-in-haystack 1B')  # Adds a title
plt.xlabel('Directory')  # X-axis label
plt.ylabel(VIEW)  # Y-axis label
plt.tight_layout()  # Fits everything neatly into the figure

# Show the plot
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories 
dirs = """experiments/llama32_1B_instruct/baseline_mnih_long_context
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/mnih_long_context
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r256/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512/eval/mnih_long_context
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-r256",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r512", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r512",
          "diff-attn-lambda=0.9-r32", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r64", "diff-attn-lambda=0.9-rightlora-r512",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        print(f"Missing {eval_file}")
        for metric in [m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Long-context multi-needle-in-haystack metric = "+metric+ " (lambda > 0.5 with groupnorm)", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_MNIH.png")
plt.show()


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import glob

# Directories

dirs = """experiments/llama32_1B_instruct/baseline_mnih_long_context
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/mnih_long_context
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r256/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512/eval/mnih_long_context
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-r256",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r512", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r512",
          "diff-attn-lambda=0.9-r32", "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r64", "diff-attn-lambda=0.9-rightlora-r512",
          ]

VIEW = "Document Depth"
# VIEW = "Context Length"
em = "EM"
m = "M"
recall = "Recall"
metric = recall
data = []

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_out.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            output = json.load(f)
            for row in output:
                # q_id = mnih_long_context_cl100_dp0
                context_length = int(row["q_id"].split("_")[3][2:])
                document_depth = int(row["q_id"].split("_")[4][2:])
                score = row[metric]
                data.append({
                    "Document Depth": document_depth,
                    "Context Length": context_length,
                    "Score": score,
                    "dir": labels[i],
                })
    else:
        print(f"Missing {eval_file}")

# plot a heatmap with y axis context length and x axis each directory (average across all document depths)
# Create a DataFrame
df = pd.DataFrame(data)
pivot_table = pd.pivot_table(df, values='Score', index=[VIEW, 'dir'], aggfunc='mean').reset_index() # This will aggregate
pivot_table = pivot_table.pivot(index=VIEW, columns="dir", values="Score") # This will turn into a proper pivot
# reorder colums in same order as labels
pivot_table = pivot_table[labels]

# Create the heatmap with better aesthetics
plt.figure(figsize=(17.5, 8))  # Can adjust these dimensions as needed
sns.heatmap(
    pivot_table,
    # annot=True,
    fmt="g",
    cmap=cmap,
    cbar_kws={'label': 'Recall'}
)

# More aesthetics
plt.title('Long-context multi-needle-in-haystack 1B')  # Adds a title
plt.xlabel('Directory')  # X-axis label
plt.ylabel(VIEW)  # Y-axis label
plt.tight_layout()  # Fits everything neatly into the figure

# Show the plot
plt.show()

In [ ]:
# after train test split in bergen training pipeline
train_distrib = {'squad': 86709, 'adversarial_qa': 29640, 'nq_open': 87041, 'hotpotqa': 87957, 'msmarco': 59111, 'triviaqa': 61192, 'freebase_qa': 20167, 'sciq': 11556, 'asqa': 4315, 'wikiqa': 804}
test_distrib = {'squad': 887, 'adversarial_qa': 326, 'nq_open': 884, 'hotpotqa': 882, 'msmarco': 588, 'triviaqa': 605, 'freebase_qa': 189, 'sciq': 123, 'asqa': 38, 'wikiqa': 9}

# plot distribs

import matplotlib.pyplot as plt
import numpy as np

# normalize by total number of elements
train_distrib = {k: v/sum(train_distrib.values()) for k,v in train_distrib.items()}
test_distrib = {k: v/sum(test_distrib.values()) for k,v in test_distrib.items()}

labels = list(train_distrib.keys())
train = list(train_distrib.values())
test = list(test_distrib.values())

x = np.arange(len(labels))  # the label locations
width = 0.35  # the width of the bars

fig, ax = plt.subplots()
rects1 = ax.bar(x - width/2, train, width, label='Train')

rects2 = ax.bar(x + width/2, test, width, label='Test')

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Number of examples')
ax.set_title('Number of examples per dataset')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=45)
ax.legend()
plt.legend()
plt.show()

In [ ]:
import datasets
ds_train = datasets.load_from_disk("datasets/BIOASQ12B_train")
ds_dev = datasets.load_from_disk("datasets/BIOASQ12B_dev")
ds = datasets.DatasetDict({"train": ds_train, "dev": ds_dev})
ds.push_to_hub('naver/bergen_bioasq12b', private=True)

In [ ]:
# huggingface login
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# open 'experiments/llama32_1B_instruct/baseline_nih_long_context/eval_dev_out.json'
import json
with open('experiments/llama32_1B_instruct/baseline_nih_long_context/eval_dev_out.json') as f:
    data = json.load(f)
    
import pandas as pd
df = pd.DataFrame(data)

# data where Recall is 0
df[df['Recall'] == 0]

In [ ]:
df[df['Recall'] == 0]['instruction'][0]

In [ ]:
df[df['Recall'] == 1]

In [ ]:
dirs = """experiments/llama32_1B_instruct/baseline_multiqa
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512", "diff-attn-lambda=0.1", "diff-attn-lambda=0.1 (no scaling)", "diff-attn-lambda=0.1-r256","diff-attn-lambda=0.1-r256 (no scaling)",
          "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.1-rightlora-r64 (no scaling)", "diff-attn-lambda=0.1-rightlora-r512", "diff-attn-lambda=0.1-rightlora-r512 (no scaling)", 
          "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r32 (no scaling)", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-r256 (no scaling)", 
          "diff-attn-lambda=0.5-rightlora-r64", "diff-attn-lambda=0.5-rightlora-r64 (no scaling)", "diff-attn-lambda=0.5-rightlora-r512","diff-attn-lambda=0.5-rightlora-r512 (no scaling)",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [em, m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [em, m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [em, m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Eval MultiQA metric = "+metric+ " with vs without post attn scaling", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_MQA_metrics_scaling_vs_nopostattnscaling.png")
plt.show()


In [ ]:
dirs = """experiments/llama32_1B_instruct/baseline_multiqa
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512",  "diff-attn-lambda=0.1 (no scaling)","diff-attn-lambda=0.1-r256 (no scaling)",
        "diff-attn-lambda=0.1-rightlora-r64 (no scaling)", "diff-attn-lambda=0.1-rightlora-r512 (no scaling)", 
        "diff-attn-lambda=0.5-r32 (no scaling)", "diff-attn-lambda=0.5-r256 (no scaling)", 
        "diff-attn-lambda=0.5-rightlora-r64 (no scaling)", "diff-attn-lambda=0.5-rightlora-r512 (no scaling)",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [em, m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [em, m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [em, m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Eval MultiQA metric = "+metric+ " without post attn scaling", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_MQA_metrics_nopostattnscaling.png")
plt.show()

In [ ]:
# compute percent improvement for directories that end with '_nopostattnscaling'
em_pc_improvement = []
match_pc_improvement = []
recall_pc_improvement = []
for i in range(3,len(dirs), 2):
    em_nopostattnscaling = data['EM'][i+1][1]
    em = data['EM'][i][1]
    percent_improvement = (em_nopostattnscaling - em) / em * 100
    em_pc_improvement.append(percent_improvement)
    print(f"{labels[i]}: {percent_improvement:.2f}% improvement in EM")

    match_nopostattnscaling = data['M'][i+1][1]
    match = data['M'][i][1]
    percent_improvement = (match_nopostattnscaling - match) / match * 100
    match_pc_improvement.append(percent_improvement)
    print(f"{labels[i]}: {percent_improvement:.2f}% improvement in Match")

    recall_nopostattnscaling = data['Recall'][i+1][1]
    recall = data['Recall'][i][1]
    percent_improvement = (recall_nopostattnscaling - recall) / recall * 100
    recall_pc_improvement.append(percent_improvement)
    print(f"{labels[i]}: {percent_improvement:.2f}% improvement in Recall")

print("Average percent improvement without post attn scaling:")
print(f"EM: {sum(em_pc_improvement)/len(em_pc_improvement):.2f}%")
print(f"Match: {sum(match_pc_improvement)/len(match_pc_improvement):.2f}%")
print(f"Recall: {sum(recall_pc_improvement)/len(recall_pc_improvement):.2f}%")

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]

dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling/eval
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512",  "diff-attn-lambda=0.1 (no scaling)","diff-attn-lambda=0.1-r256 (no scaling)",
        "diff-attn-lambda=0.1-rightlora-r64 (no scaling)", "diff-attn-lambda=0.1-rightlora-r512 (no scaling)", 
        "diff-attn-lambda=0.5-r32 (no scaling)", "diff-attn-lambda=0.5-r256 (no scaling)", 
        "diff-attn-lambda=0.5-rightlora-r64 (no scaling)", "diff-attn-lambda=0.5-rightlora-r512 (no scaling)",
          ]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# metric = "EM"
# metric = "M"
metric = "Recall"
metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

# Plotting
num_domains = len(domains)
fig, axes = plt.subplots(num_domains, 1, figsize=(8, 7 * num_domains))

for i, (ax, (domain, values)) in enumerate(zip(axes, recall_data.items())):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, recalls, color=colors[i])
    ax.set_title(domain + " " + metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_MultiQA_eval_domains_{metric}_nopostattnscaling.png")
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains

dirs = """experiments/llama32_1B_instruct/baseline_nih_long_context
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/nih_long_context
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling/eval/nih_long_context
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512",  "diff-attn-lambda=0.1 (no scaling)","diff-attn-lambda=0.1-r256 (no scaling)",
        "diff-attn-lambda=0.1-rightlora-r64 (no scaling)", "diff-attn-lambda=0.1-rightlora-r512 (no scaling)", 
        "diff-attn-lambda=0.5-r32 (no scaling)", "diff-attn-lambda=0.5-r256 (no scaling)", 
        "diff-attn-lambda=0.5-rightlora-r64 (no scaling)", "diff-attn-lambda=0.5-rightlora-r512 (no scaling)",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Long-context needle-in-haystack metric = "+metric+ " (lambda > 0.5 with groupnorm)", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_NIH_nopostattnscaling.png")
plt.show()


In [ ]:
'''
code for plot taken from https://github.com/gkamradt/LLMTest_NeedleInAHaystack/blob/main/viz/CreateVizFromLLMTesting.ipynb
'''

import os
import json
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

dirs = """experiments/llama32_1B_instruct/baseline_nih_long_context
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/nih_long_context
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling/eval/nih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling/eval/nih_long_context
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512",  "diff-attn-lambda=0.1 (no scaling)","diff-attn-lambda=0.1-r256 (no scaling)",
        "diff-attn-lambda=0.1-rightlora-r64 (no scaling)", "diff-attn-lambda=0.1-rightlora-r512 (no scaling)", 
        "diff-attn-lambda=0.5-r32 (no scaling)", "diff-attn-lambda=0.5-r256 (no scaling)", 
        "diff-attn-lambda=0.5-rightlora-r64 (no scaling)", "diff-attn-lambda=0.5-rightlora-r512 (no scaling)",
          ]

VIEW = "Document Depth"
# VIEW = "Context Length"
em = "EM"
m = "M"
recall = "Recall"
metric = recall
data = []

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_out.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            output = json.load(f)
            for row in output:
                # q_id = mnih_long_context_cl100_dp0
                context_length = int(row["q_id"].split("_")[3][2:])
                document_depth = int(row["q_id"].split("_")[4][2:])
                score = row[metric]
                data.append({
                    "Document Depth": document_depth,
                    "Context Length": context_length,
                    "Score": score,
                    "dir": labels[i],
                })
    else:
        print(f"Missing {eval_file}")

# plot a heatmap with y axis context length and x axis each directory (average across all document depths)
# Create a DataFrame
cmap = LinearSegmentedColormap.from_list("custom_cmap", ["#F0496E", "#EBB839", "#0CD79F"])

df = pd.DataFrame(data)
pivot_table = pd.pivot_table(df, values='Score', index=[VIEW, 'dir'], aggfunc='mean').reset_index() # This will aggregate
pivot_table = pivot_table.pivot(index=VIEW, columns="dir", values="Score") # This will turn into a proper pivot
# reorder colums in same order as labels
pivot_table = pivot_table[labels]

# Create the heatmap with better aesthetics
plt.figure(figsize=(17.5, 8))  # Can adjust these dimensions as needed
sns.heatmap(
    pivot_table,
    # annot=True,
    fmt="g",
    cmap=cmap,
    cbar_kws={'label': 'Recall'}
)

# More aesthetics
plt.title('Long-context needle-in-haystack 1B')  # Adds a title
plt.xlabel('Directory')  # X-axis label
plt.ylabel(VIEW)  # Y-axis label
plt.tight_layout()  # Fits everything neatly into the figure

# Show the plot
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains

dirs = """experiments/llama32_1B_instruct/baseline_mnih_long_context
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/mnih_long_context
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling/eval/mnih_long_context
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512",  "diff-attn-lambda=0.1 (no scaling)","diff-attn-lambda=0.1-r256 (no scaling)",
        "diff-attn-lambda=0.1-rightlora-r64 (no scaling)", "diff-attn-lambda=0.1-rightlora-r512 (no scaling)", 
        "diff-attn-lambda=0.5-r32 (no scaling)", "diff-attn-lambda=0.5-r256 (no scaling)", 
        "diff-attn-lambda=0.5-rightlora-r64 (no scaling)", "diff-attn-lambda=0.5-rightlora-r512 (no scaling)",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Long-context multi-needle-in-haystack metric = "+metric+ " (lambda > 0.5 with groupnorm)", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_MNIH_nopostattnscaling.png")
plt.show()


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap
import glob

# Directories

dirs = """experiments/llama32_1B_instruct/baseline_mnih_long_context
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/mnih_long_context
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling/eval/mnih_long_context
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling/eval/mnih_long_context
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r512",  "diff-attn-lambda=0.1 (no scaling)","diff-attn-lambda=0.1-r256 (no scaling)",
        "diff-attn-lambda=0.1-rightlora-r64 (no scaling)", "diff-attn-lambda=0.1-rightlora-r512 (no scaling)", 
        "diff-attn-lambda=0.5-r32 (no scaling)", "diff-attn-lambda=0.5-r256 (no scaling)", 
        "diff-attn-lambda=0.5-rightlora-r64 (no scaling)", "diff-attn-lambda=0.5-rightlora-r512 (no scaling)",
          ]

VIEW = "Document Depth"
VIEW = "Context Length"
em = "EM"
m = "M"
recall = "Recall"
metric = recall
data = []

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_out.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            output = json.load(f)
            for row in output:
                # q_id = mnih_long_context_cl100_dp0
                context_length = int(row["q_id"].split("_")[3][2:])
                document_depth = int(row["q_id"].split("_")[4][2:])
                score = row[metric]
                data.append({
                    "Document Depth": document_depth,
                    "Context Length": context_length,
                    "Score": score,
                    "dir": labels[i],
                })
    else:
        print(f"Missing {eval_file}")

# plot a heatmap with y axis context length and x axis each directory (average across all document depths)
# Create a DataFrame
df = pd.DataFrame(data)
pivot_table = pd.pivot_table(df, values='Score', index=[VIEW, 'dir'], aggfunc='mean').reset_index() # This will aggregate
pivot_table = pivot_table.pivot(index=VIEW, columns="dir", values="Score") # This will turn into a proper pivot
# reorder colums in same order as labels
pivot_table = pivot_table[labels]

# Create the heatmap with better aesthetics
plt.figure(figsize=(17.5, 8))  # Can adjust these dimensions as needed
sns.heatmap(
    pivot_table,
    # annot=True,
    fmt="g",
    cmap=cmap,
    cbar_kws={'label': 'Recall'}
)

# More aesthetics
plt.title('Long-context multi-needle-in-haystack 1B')  # Adds a title
plt.xlabel('Directory')  # X-axis label
plt.ylabel(VIEW)  # Y-axis label
plt.tight_layout()  # Fits everything neatly into the figure

# Show the plot
plt.show()

In [ ]:
dirs = """experiments/llama32_1B_instruct/baseline_multiqa
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_Lora_r32qkvo_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_Lora_r256qkvo_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_loraVO
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_loraVO
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_loraVO
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_loraVO
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64_loraVO
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r256
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512_loraVO
""".split()

labels = ["llama", "llama-lora-r64", "llama-lora-r32-VO", "llama-lora-r512", "llama-lora-r256-VO",
          "diff-attn-lambda=0.1", "diff-attn-lambda=0.1-rightlora-r64","diff-attn-lambda=0.1-r32-rightlora-VO",
          "diff-attn-lambda=0.1-r256", "diff-attn-lambda=0.1-rightlora-r512", "diff-attn-lambda=0.1-r256-rightlora-VO",
          "diff-attn-lambda=0.5", "diff-attn-lambda=0.5-rightlora-r64","diff-attn-lambda=0.5-r32-rightlora-VO",
          "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r512", "diff-attn-lambda=0.5-r256-rightlora-VO",
          "diff-attn-lambda=0.9", "diff-attn-lambda=0.9-rightlora-r64","diff-attn-lambda=0.9-r32-rightlora-VO",
          "diff-attn-lambda=0.9-r256", "diff-attn-lambda=0.9-rightlora-r512", "diff-attn-lambda=0.9-r256-rightlora-VO",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [em, m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [em, m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [em, m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Eval MultiQA metric = "+metric+ " (lambda > 0.5 with groupnorm) (compare lora VO)", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_MQA_metrics_loraVO.png")
plt.show()


In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]

dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_r32qkvo_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_Lora_r256qkvo_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_loraVO/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_loraVO/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_loraVO/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_loraVO/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64_loraVO/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512_loraVO/eval
""".split()

labels = ["llama", "llama-lora-r32-VO", "llama-lora-r256-VO",
          "diff-attn-lambda=0.1-r32-rightlora-VO",
          "diff-attn-lambda=0.1-r256-rightlora-VO",
          "diff-attn-lambda=0.5-r32-rightlora-VO",
          "diff-attn-lambda=0.5-r256-rightlora-VO",
          "diff-attn-lambda=0.9-r32-rightlora-VO",
          "diff-attn-lambda=0.9-r256-rightlora-VO",
          ]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# metric = "EM"
# metric = "M"
metric = "Recall"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

# Plotting
num_domains = len(domains)
fig, axes = plt.subplots(num_domains, 1, figsize=(8, 7 * num_domains))

for i, (ax, (domain, values)) in enumerate(zip(axes, recall_data.items())):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, recalls, color=colors[i])
    ax.set_title(domain + " " + metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_loraVO_MultiQA_eval_domains_{metric}.png")
plt.show()

In [ ]:
dirs = """experiments/llama32_1B_instruct/baseline_multiqa
experiments/llama32_1B_instruct/train_Lora_r32qkvomlp_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_Lora_r256qkvomlp_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_loraVOMLP
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_loraVOMLP
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_loraVOMLP
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_loraVOMLP
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64_loraVOMLP
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512_loraVOMLP
""".split()

labels = ["llama", "llama-lora-r32-VOMLP", "llama-lora-r256-VOMLP",
          "diff-attn-lambda=0.1-r32-rightlora-VOMLP",
            "diff-attn-lambda=0.1-r256-rightlora-VOMLP",
          "diff-attn-lambda=0.5-r32-rightlora-VOMLP",
        "diff-attn-lambda=0.5-r256-rightlora-VOMLP",
         "diff-attn-lambda=0.9-r32-rightlora-VOMLP",
         "diff-attn-lambda=0.9-r256-rightlora-VOMLP",
          ]

em = "EM"
m = "M"
recall = "Recall"
# Initialize data storage
data = {metric: [] for metric in [em, m, recall]}

for i,dir in enumerate(dirs):
    eval_file = os.path.join(dir, "eval_dev_metrics.json")
    if os.path.exists(eval_file):
        with open(eval_file, "r") as f:
            metrics = json.load(f)
            for metric in [em, m, recall]:
                value = metrics.get(metric, 0)
                data[metric].append((labels[i], value))
    else:
        for metric in [em, m, recall]:
            data[metric].append((labels[i], None))
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Plotting
num_metrics = len(data)
fig, axes = plt.subplots(num_metrics, 1, figsize=(12, 7 * num_metrics))

for i, (ax, (metric, values)) in enumerate(zip(axes, data.items())):
    dirs, metrics = zip(*values)
    metrics = [m if m is not None else 0 for m in metrics]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, metrics, color=colors[i])
    ax.set_title("Eval MultiQA metric = "+metric+ " (lambda > 0.5 with groupnorm) (compare lora VO)", fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_eval_MQA_metrics_loraVOMLP.png")
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]

dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_r32qkvomlp_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_Lora_r256qkvomlp_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_loraVOMLP/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_loraVOMLP/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_loraVOMLP/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_loraVOMLP/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r64_loraVOMLP/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.9_r512_loraVOMLP/eval
""".split()

labels = ["llama", "llama-lora-r32-VOMLP", "llama-lora-r256-VOMLP",
          "diff-attn-lambda=0.1-r32-rightlora-VOMLP",
            "diff-attn-lambda=0.1-r256-rightlora-VOMLP",
          "diff-attn-lambda=0.5-r32-rightlora-VOMLP",
        "diff-attn-lambda=0.5-r256-rightlora-VOMLP",
         "diff-attn-lambda=0.9-r32-rightlora-VOMLP",
         "diff-attn-lambda=0.9-r256-rightlora-VOMLP",
          ]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# metric = "EM"
# metric = "M"
metric = "Recall"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

# Plotting
num_domains = len(domains)
fig, axes = plt.subplots(num_domains, 1, figsize=(8, 7 * num_domains))

for i, (ax, (domain, values)) in enumerate(zip(axes, recall_data.items())):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, recalls, color=colors[i])
    ax.set_title(domain + " " + metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_loraVOMLP_MultiQA_eval_domains_{metric}.png")
plt.show()

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa", "nih_long_context", "mnih_long_context"
]

dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval
experiments/llama32_1B_instruct/IFT_llama_32_1B_lora_r64_qk_tulu3_MERGED/eval_top5docs
experiments/llama32_1B_instruct/IFT_llama_32_1B_lamb_0.1_r64_tulu3_fix/eval
""".split()

labels = ["Llama 3.2 1B Instruct", 
          "LoRA FT-MultiQA r64",
          "DiffLoRA FT-MultiQA lambda=0.1 r64", 
          "LoRA IFT-tulu3 r64",
          "DiffLoRA IFT-tulu3 lambda=0.1 r64",
          ]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]

# metric = "EM"
# metric = "M"
metric = "Recall"
# metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

# Plotting
num_domains = len(domains)
fig, axes = plt.subplots(num_domains, 1, figsize=(8, 7 * num_domains))

for i, (ax, (domain, values)) in enumerate(zip(axes, recall_data.items())):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0
    
    x_positions = np.arange(len(dirs))
    ax.bar(x_positions, recalls, color=colors[i])
    ax.set_title(domain + " " + metric, fontsize=14, fontweight='bold', color=colors[i])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=8)
    ax.grid(axis='y')

axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/FT_vs_IFT_eval_domains_{metric}.png")
plt.show()

In [ ]:
import json
with open('experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval/syllabusqa/eval_dev_out.json', 'r') as f:
    data_top5docs = json.load(f)
with open('experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval_top10docs/syllabusqa/eval_dev_out.json', 'r') as f:
    data_top10docs = json.load(f)
with open('experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval_top20docs/syllabusqa/eval_dev_out.json', 'r') as f:
    data_top20docs = json.load(f)

for data in [data_top5docs, data_top10docs, data_top20docs]:
    count_course_code_in_docs = 0
    for row in data:
        course_code = row["question"].split(":")[0]
        if course_code in row["instruction"].split("### The question is:")[0]:
            count_course_code_in_docs += 1
    total_rows = len(data)
    print(count_course_code_in_docs, total_rows, count_course_code_in_docs/total_rows)

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

TOP_K = 20

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]

dirs = f"""experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling/eval_top{TOP_K}docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling/eval_top{TOP_K}docs
""".split()

labels = [
    "llama-lora-r64", "diff-attn-lambda=0.1-r32", "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-rightlora-r64",
    "llama-lora-r512", "diff-attn-lambda=0.1-r256", "diff-attn-lambda=0.1-rightlora-r512", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r512",
]

hatches = ["*", "/", "/", "/", "/", "*", "/", "/", "/", "/"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# metric = "EM"
# metric = "M"
metric = "Recall"
metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    ax.set_title(domain + " TOP_K=" + str(TOP_K) + " " + metric, fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_MultiQA_eval_domains_{metric}_top{TOP_K}.png")

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]

dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5_4distractors
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_nopostattnscaling/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_nopostattnscaling/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_nopostattnscaling/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_nopostattnscaling/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_nopostattnscaling/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_nopostattnscaling/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_nopostattnscaling/eval_top5docs_4distractors
""".split()

labels = [
    "llama",
    "llama-lora-r64", "diff-attn-lambda=0.1-r32", "diff-attn-lambda=0.1-rightlora-r64", "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-rightlora-r64",
    "llama-lora-r512", "diff-attn-lambda=0.1-r256", "diff-attn-lambda=0.1-rightlora-r512", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-rightlora-r512",
]

hatches = ["O", "*", "/", "/", "/", "/", "*", "/", "/", "/", "/"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# metric = "EM"
# metric = "M"
metric = "Recall"
metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

        
num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    ax.set_title(domain + " " + metric, fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

# axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_MultiQA_eval_domains_{metric}_4distractors.png")

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa", "nih_long_context", "mnih_long_context"
]

dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval
experiments/llama32_1B_instruct/IFT_llama_32_1B_lora_r64_qk_tulu3_MERGED/eval_top5docs
experiments/llama32_1B_instruct/IFT_llama_32_1B_lamb_0.1_r64_tulu3_fix/eval
experiments/llama32_1B_instruct/IFT_llama_32_1B_lamb_0.5_r64_tulu3_fix/eval
experiments/llama32_1B_instruct/IFT_llama_32_1B_lamb_0.9_r64_tulu3_fix/eval
""".split()

labels = ["Llama 3.2 1B Instruct", 
          "LoRA FT-MultiQA r64",
          "DiffLoRA FT-MultiQA lambda=0.1 r64", 
          "LoRA IFT-tulu3 r64",
          "DiffLoRA IFT-tulu3 lambda=0.1 r64",
            "DiffLoRA IFT-tulu3 lambda=0.5 r64",
            "DiffLoRA IFT-tulu3 lambda=0.9 r64",
          ]

hatches = ["O", "*", "/", "*", "/", "/", "/"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# metric = "EM"
# metric = "M"
metric = "Recall"
metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

        
num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    ax.set_title(domain + " " + metric, fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

# axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_IFT_eval_domains_{metric}.png")

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]

dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5_4distractors
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_nopostattnscaling/eval_top5docs_4distractors
experiments/llama32_1B_instruct/IFT_llama_32_1B_lora_r64_qk_tulu3_MERGED/eval_top5docs_4distractors
experiments/llama32_1B_instruct/IFT_llama_32_1B_lamb_0.1_r64_tulu3_fix/eval_top5docs_4distractors
experiments/llama32_1B_instruct/IFT_llama_32_1B_lamb_0.5_r64_tulu3_fix/eval_top5docs_4distractors
experiments/llama32_1B_instruct/IFT_llama_32_1B_lamb_0.9_r64_tulu3_fix/eval_top5docs_4distractors
""".split()

labels = ["Llama 3.2 1B Instruct", 
          "LoRA FT-MultiQA r64",
          "DiffLoRA FT-MultiQA lambda=0.1 r64", 
          "LoRA IFT-tulu3 r64",
          "DiffLoRA IFT-tulu3 lambda=0.1 r64",
            "DiffLoRA IFT-tulu3 lambda=0.5 r64",
            "DiffLoRA IFT-tulu3 lambda=0.9 r64",
          ]

hatches = ["O", "*", "/", "*", "/", "/", "/"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# metric = "EM"
# metric = "M"
metric = "Recall"
metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

        
num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    ax.set_title(domain + " " + metric, fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

# axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_IFT_eval_domains_{metric}_4distractors.png")

#### Does performance actually decrease with distractors for lora? Yes

In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa"
]

dirs = """experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top5docs_4distractors
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top5docs_4distractors
""".split()

labels = [
    "llama-lora-r64-top5docs", "llama-lora-r64-top5docs-4distractors",
    "llama-lora-r512-top5docs", "llama-lora-r512-top5docs-4distractors",
]

hatches = ["*", "/", "*", "/"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# metric = "EM"
# metric = "M"
metric = "Recall"
metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,dir in enumerate(dirs):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))

        
num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    ax.set_title(domain + " " + metric, fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

# axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
# plt.savefig(f"figs/diff_attn_1B_MultiQA_eval_domains_{metric}_4distractors.png")

## Multid eval

In [ ]:
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "multiqa", "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa", "nih_long_context", "mnih_long_context"
]

eval_dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_Lora_r512_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r256_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r512_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r256_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r512_noscalingpostattn_negativetermloraonly/eval
""".split()

multiqa_dirs = ["experiments/llama32_1B_instruct/baseline_multiqa"] + [Path(eval_dir).parent for eval_dir in eval_dirs[1:]]

labels = [
    "llama",
    "llama-lora-r64", "diff-attn-lambda=0.1-r32", "diff-attn-lambda=0.1-r64", "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r64", 
    "llama-lora-r512", "diff-attn-lambda=0.1-r256", "diff-attn-lambda=0.1-r512", "diff-attn-lambda=0.5-r256", "diff-attn-lambda=0.5-r512", 
]

hatches = ["O", "*", "/", "/", "/", "/", "*", "/", "/", "/", "/"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]


em = "EM"
m = "M"
recall = "Recall"
metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,(multiqa_dir,dir) in enumerate(zip(multiqa_dirs,eval_dirs)):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json") if domain != "multiqa" else os.path.join(multiqa_dir, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                if domain == "multiqa":
                    recall = metrics.get("Recall", 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))


num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    if domain == "multiqa":
        ax.set_title(domain + " " + "Recall", fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    else:
        ax.set_title(domain.removeprefix("robustqa_").removesuffix("_long_context") + " " + metric, fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

# axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_negativetermloraonly_MultiQA_eval_domains_{metric}.png")


## Compared to negative_term_full_dim

In [ ]:
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "multiqa", "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa", "nih_long_context", "mnih_long_context"
]

eval_dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top3
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermfulldim/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermfulldim/eval
""".split()

multiqa_dirs = ["experiments/llama32_1B_instruct/baseline_multiqa"] + [Path(eval_dir).parent for eval_dir in eval_dirs[1:]]

labels = [
    "llama",
    "llama-lora-r64", "diff-lora-lmb0.1-r32-rhs=loraonly", "diff-lora-lmb0.1-r64-rhs=loraonly", "diff-lora-lmb0.1-r64-rhs=fulldim",
    "diff-lora-lmb0.5-r32-rhs=loraonly", "diff-lora-lmb0.5-r64-rhs=loraonly", "diff-lora-lmb0.5-r64-rhs=fulldim",
]

hatches = ["O", "*", "/", "/", "+", "/", "/", "+"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]


em = "EM"
m = "M"
recall = "Recall"
metric = "LLMeval_vllm_SOLAR-107B"
# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,(multiqa_dir,dir) in enumerate(zip(multiqa_dirs,eval_dirs)):
    for domain in domains:
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json") if domain != "multiqa" else os.path.join(multiqa_dir, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                if domain == "multiqa":
                    recall = metrics.get("Recall", 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))


num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    if domain == "multiqa":
        ax.set_title(domain + " " + "Recall", fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    else:
        ax.set_title(domain.removeprefix("robustqa_").removesuffix("_long_context") + " " + metric, fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

# axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_loraonly_vs_fulldim_MultiQA_eval_domains_{metric}.png")

## Compare multid scores of standard difflora and distillation difflora

In [ ]:
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "multiqa", "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa", "nih_long_context", "mnih_long_context"
]

domains_metrics = [
    "Recall", "M", "M", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "M", "Recall"
]

assert len(domains) == len(domains_metrics)

eval_dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_llama321b_instruct_spladeberta_top3_basicprompt/eval_top5docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_Lora_MultiQA_distillmistral7b_llama321b_instruct_spladeberta_top5_basicprompt/eval_top5docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.1_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.5_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermloraonly/eval
""".split()

multiqa_dirs = ["experiments/llama32_1B_instruct/baseline_multiqa"] + [Path(eval_dir).parent for eval_dir in eval_dirs[1:]]

labels = [
    "llama",
    "llama-lora-r64", "diff-attn-lambda=0.1-r32", "diff-attn-lambda=0.1-r64", "diff-attn-lambda=0.5-r32", "diff-attn-lambda=0.5-r64",
    "llama-lora-r64-distill", "diff-attn-lambda=0.1-r32-distill", "diff-attn-lambda=0.1-r64-distill", "diff-attn-lambda=0.5-r32-distill", "diff-attn-lambda=0.5-r64-distill",
]

hatches = ["O", "*", "/", "/", "/", "/", "*", "/", "/", "/", "/"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,(multiqa_dir,dir) in enumerate(zip(multiqa_dirs,eval_dirs)):
    for domain,metric in zip(domains,domains_metrics):
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json") if domain != "multiqa" else os.path.join(multiqa_dir, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))


num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    ax.set_title(domain.removeprefix("robustqa_").removesuffix("_long_context") + " " + domains_metrics[i], fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

# axes[-1].set_xlabel("Directories", fontsize=12)
plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_MultiQAdistill_eval_domains_{metric}.png")


## Compare with negative term full dim

In [ ]:
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "multiqa", "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa", "nih_long_context", "mnih_long_context"
]

domains_metrics = [
    "Recall", "M", "M", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "M", "Recall"
]

assert len(domains) == len(domains_metrics)

eval_dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_distillmistral7b_llama321b_instruct_spladeberta_top5_basicprompt/eval_top5docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.1_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermfulldim/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.5_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermfulldim/eval
""".split()

multiqa_dirs = ["experiments/llama32_1B_instruct/baseline_multiqa"] + [Path(eval_dir).parent for eval_dir in eval_dirs[1:]]

labels = [
    "llama",
    "llama-lora-r64", "diff-lora-lmb0.1-r32-rhs=loraonly", "diff-lora-lmb0.1-r64-rhs=loraonly", "diff-lora-lmb0.1-rhs=fulldim", 
    "diff-lora-lmb0.5-r32-rhs=loraonly", "diff-lora-lmb0.5-r64-rhs=loraonly", "diff-lora-lmb0.5-rhs=fulldim", 
]

hatches = ["O", "*", "/", "/", "+", "/", "/", "+"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,(multiqa_dir,dir) in enumerate(zip(multiqa_dirs,eval_dirs)):
    for domain,metric in zip(domains,domains_metrics):
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json") if domain != "multiqa" else os.path.join(multiqa_dir, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))


num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    ax.set_title(domain.removeprefix("robustqa_").removesuffix("_long_context") + " " + domains_metrics[i], fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_loraonly_vs_fulldim_MultiQAdistill_eval_domains_{metric}.png")

## Adding ReLU

In [ ]:
import os
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
from math import ceil, sqrt

# Directories and domains
domains = [
    "multiqa", "nq", "popqa",
    "bioasq12b", "covidqa", "fiqa", "paraphraserc", "robustqa_lifestyle", 
    "robustqa_recreation", "robustqa_science", "robustqa_technology", "robustqa_writing", 
    "searchqa", "syllabusqa", "techqa", "nih_long_context", "mnih_long_context"
]

domains_metrics = [
    "Recall", "M", "M", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "LLMeval_vllm_SOLAR-107B", "M", "Recall"
]

assert len(domains) == len(domains_metrics)

eval_dirs = """experiments/control_nq_llama/llama321binstruct_evaltop5
experiments/llama32_1B_instruct/train_Lora_MultiQA_distillmistral7b_llama321b_instruct_spladeberta_top5_basicprompt/eval_top5docs
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.1_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.1_r64_noscalingpostattn_negativetermfulldim/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.5_r32_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermloraonly/eval
experiments/llama32_1B_instruct/train_LoraDiffAtt_multiqa_distillmistral7b_spladeberta_top5_0.5_r64_noscalingpostattn_negativetermfulldim/eval
""".split()

multiqa_dirs = ["experiments/llama32_1B_instruct/baseline_multiqa"] + [Path(eval_dir).parent for eval_dir in eval_dirs[1:]]

labels = [
    "llama",
    "llama-lora-r64", "diff-lora-lmb0.1-r32-rhs=loraonly", "diff-lora-lmb0.1-r64-rhs=loraonly", "diff-lora-lmb0.1-rhs=fulldim", 
    "diff-lora-lmb0.5-r32-rhs=loraonly", "diff-lora-lmb0.5-r64-rhs=loraonly", "diff-lora-lmb0.5-rhs=fulldim", 
]

hatches = ["O", "*", "/", "/", "+", "/", "/", "+"]

colors = ["#2ca02c", "#d62728", "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf", "#1f77b4", "#ff7f0e"]

# Initialize data storage
recall_data = {domain: [] for domain in domains}

# Extract Recall values for each domain
for i,(multiqa_dir,dir) in enumerate(zip(multiqa_dirs,eval_dirs)):
    for domain,metric in zip(domains,domains_metrics):
        eval_file = os.path.join(dir, domain, "eval_dev_metrics.json") if domain != "multiqa" else os.path.join(multiqa_dir, "eval_dev_metrics.json")
        if os.path.exists(eval_file):
            with open(eval_file, "r") as f:
                metrics = json.load(f)
                recall = metrics.get(metric, 0)
                recall_data[domain].append((labels[i], recall))
        else:
            print(f"Missing {eval_file}")
            recall_data[domain].append((labels[i], None))


num_domains = len(domains)
grid_size = ceil(sqrt(num_domains))
fig, axes = plt.subplots(grid_size, grid_size, figsize=(64, 64))
axes = axes.flatten()

for i, (domain, values) in enumerate(recall_data.items()):
    dirs, recalls = zip(*values)
    recalls = [r if r is not None else 0 for r in recalls]  # Replace None with 0

    x_positions = np.arange(len(dirs))
    ax = axes[i]
    for x_pos in range(len(x_positions)):
        ax.bar(x_positions[x_pos], recalls[x_pos], color=colors[i % len(colors)], hatch=hatches[x_pos])
    for x, y in zip(x_positions, recalls):
        ax.text(x, y + 0.001, f"{y:.3f}", ha='center', va='bottom', fontsize=28)
    ax.set_title(domain.removeprefix("robustqa_").removesuffix("_long_context") + " " + domains_metrics[i], fontsize=40, fontweight='bold', color=colors[i % len(colors)])
    ax.set_ylabel(metric, fontsize=12)
    ax.set_xticks(x_positions)
    ax.set_xticklabels(dirs, rotation=90, fontsize=32)
    ax.grid(axis='y')

# Hide unused subplots
for j in range(num_domains, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()

# Save the plot
plt.savefig(f"figs/diff_attn_1B_MultiQAdistill_eval_domains_{metric}_RELU.png")

# Benchmarks

## Top-k docs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import json

llama_dirs = """
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_0s
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top1
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top2
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top3
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top10
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top20
""".split()

mistral_dirs = """
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_0s
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top1
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top2
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top3
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top10
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top20
""".split()

solar_dirs = """
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_0s
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top1
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top2
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top3
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top10
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top20
""".split()

labels = ["0-shot", "top-1", "top-2", "top-3", "top-5", "top-10", "top-20"]

recalls = {
    "llama": [],
    "mistral": [],
    "solar": []
}

llmevals = {
    "llama": [],
    "mistral": [],
    "solar": []
}

# Load data
for llama_dir, mistral_dir, solar_dir in zip(llama_dirs, mistral_dirs, solar_dirs):
    with open(f"{llama_dir}/eval_dev_metrics.json", "r") as f:
        llama_metrics = json.load(f)
        recalls["llama"].append(llama_metrics.get("Recall", 0))
        llmevals["llama"].append(llama_metrics.get("LLMeval_vllm_SOLAR-107B", 0))

    with open(f"{mistral_dir}/eval_dev_metrics.json", "r") as f:
        mistral_metrics = json.load(f)
        recalls["mistral"].append(mistral_metrics.get("Recall", 0))
        llmevals["mistral"].append(mistral_metrics.get("LLMeval_vllm_SOLAR-107B", 0))

    with open(f"{solar_dir}/eval_dev_metrics.json", "r") as f:
        solar_metrics = json.load(f)
        recalls["solar"].append(solar_metrics.get("Recall", 0))
        llmevals["solar"].append(solar_metrics.get("LLMeval_vllm_SOLAR-107B", 0))

# Define plot parameters
x = np.arange(len(recalls["llama"]))
width = 0.25
# textures = ['/', '\\', '*']  # Textures for bars
textures = [None, None, None]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Recall
ax1 = axes[0]
rects1 = ax1.bar(x - width, recalls["llama"], width, label='LLAMA-1B', color='steelblue', hatch=textures[0])
rects2 = ax1.bar(x, recalls["mistral"], width, label='MISTRAL-7B', color='seagreen', hatch=textures[1])
rects3 = ax1.bar(x + width, recalls["solar"], width, label='SOLAR-10.7B', color='salmon', hatch=textures[2])
ax1.set_ylabel('Recall')
ax1.set_title('Recall by top-k')
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.legend()
for rect, data_key in zip([rects1, rects2, rects3], ["llama", "mistral", "solar"]):
    for i, rect_bar in enumerate(rect):
        ax1.text(rect_bar.get_x() + rect_bar.get_width() / 2.0, rect_bar.get_height(),
                 f"{recalls[data_key][i]:.3f}", ha='center', va='bottom', fontsize=6)

# Plot LLMeval
ax2 = axes[1]
rects4 = ax2.bar(x - width, llmevals["llama"], width, label='LLAMA-1B', color='steelblue', hatch=textures[0])
rects5 = ax2.bar(x, llmevals["mistral"], width, label='MISTRAL-7B', color='seagreen', hatch=textures[1])
rects6 = ax2.bar(x + width, llmevals["solar"], width, label='SOLAR-10.7B', color='salmon', hatch=textures[2])
ax2.set_ylabel('LLMeval')
ax2.set_title('LLMeval by top-k')
ax2.set_xticks(x)
ax2.set_xticklabels(labels)
ax2.legend()
for rect, data_key in zip([rects4, rects5, rects6], ["llama", "mistral", "solar"]):
    for i, rect_bar in enumerate(rect):
        ax2.text(rect_bar.get_x() + rect_bar.get_width() / 2.0, rect_bar.get_height(),
                 f"{llmevals[data_key][i]:.3f}", ha='center', va='bottom', fontsize=6)

plt.tight_layout()
plt.savefig('figs/benchmarks_topk.png')
plt.show()


## Distractors

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import json

llama_dirs = """
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_0s
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors5
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors4
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors3
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors2
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors1
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5
""".split()

mistral_dirs = """
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_0s
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors5
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors4
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors3
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors2
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors1
experiments/benchmarks/MISTRAL_7B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5
""".split()

solar_dirs = """
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_0s
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors5
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors4
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors3
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors2
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5_distractors1
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta_top5
""".split()

labels = ["0-shot", "5 distractors", "top-1 + 4 distractors", "top-2 + 3 distractors", "top-3 + 2 distractors", "top-4 + 1 distractor", "top-5"]

recalls = {
    "llama": [],
    "mistral": [],
    "solar": []
}

llmevals = {
    "llama": [],
    "mistral": [],
    "solar": []
}

# Load data
for llama_dir, mistral_dir, solar_dir in zip(llama_dirs, mistral_dirs, solar_dirs):
    with open(f"{llama_dir}/eval_dev_metrics.json", "r") as f:
        llama_metrics = json.load(f)
        recalls["llama"].append(llama_metrics.get("Recall", 0))
        llmevals["llama"].append(llama_metrics.get("LLMeval_vllm_SOLAR-107B", 0))

    with open(f"{mistral_dir}/eval_dev_metrics.json", "r") as f:
        mistral_metrics = json.load(f)
        recalls["mistral"].append(mistral_metrics.get("Recall", 0))
        llmevals["mistral"].append(mistral_metrics.get("LLMeval_vllm_SOLAR-107B", 0))

    with open(f"{solar_dir}/eval_dev_metrics.json", "r") as f:
        solar_metrics = json.load(f)
        recalls["solar"].append(solar_metrics.get("Recall", 0))
        llmevals["solar"].append(solar_metrics.get("LLMeval_vllm_SOLAR-107B", 0))

# Define plot parameters
x = np.arange(len(recalls["llama"]))
width = 0.25
# textures = ['/', '\\', '*']  # Textures for bars
textures = [None, None, None]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Recall
ax1 = axes[0]
rects1 = ax1.bar(x - width, recalls["llama"], width, label='LLAMA-1B', color='steelblue', hatch=textures[0])
rects2 = ax1.bar(x, recalls["mistral"], width, label='MISTRAL-7B', color='seagreen', hatch=textures[1])
rects3 = ax1.bar(x + width, recalls["solar"], width, label='SOLAR-10.7B', color='salmon', hatch=textures[2])
ax1.set_ylabel('Recall')
ax1.set_title('Recall distractors')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=90)
ax1.legend()
for rect, data_key in zip([rects1, rects2, rects3], ["llama", "mistral", "solar"]):
    for i, rect_bar in enumerate(rect):
        ax1.text(rect_bar.get_x() + rect_bar.get_width() / 2.0, rect_bar.get_height(),
                 f"{recalls[data_key][i]:.3f}", ha='center', va='bottom', fontsize=6)

# Plot LLMeval
ax2 = axes[1]
rects4 = ax2.bar(x - width, llmevals["llama"], width, label='LLAMA-1B', color='steelblue', hatch=textures[0])
rects5 = ax2.bar(x, llmevals["mistral"], width, label='MISTRAL-7B', color='seagreen', hatch=textures[1])
rects6 = ax2.bar(x + width, llmevals["solar"], width, label='SOLAR-10.7B', color='salmon', hatch=textures[2])
ax2.set_ylabel('LLMeval')
ax2.set_title('LLMeval distractors')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=90)
ax2.legend()
for rect, data_key in zip([rects4, rects5, rects6], ["llama", "mistral", "solar"]):
    for i, rect_bar in enumerate(rect):
        ax2.text(rect_bar.get_x() + rect_bar.get_width() / 2.0, rect_bar.get_height(),
                 f"{llmevals[data_key][i]:.3f}", ha='center', va='bottom', fontsize=6)

plt.tight_layout()
plt.savefig('figs/benchmarks_distractors.png')
plt.show()

## Rerankers

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import json

llama_dirs = """
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRminilm6
experiments/benchmarks/LLAMA_32_1B_INSTRUCT/bioasq12b_RETsplade_RRdeberta
""".split()

solar_dirs = """
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRminilm6
experiments/benchmarks/SOLAR_107B_INSTRUCT/bioasq12b_RETsplade_RRdeberta
""".split()

labels = ["No RR", "RR MiniLM6", "RR DeBERTa-v3"]

recalls = {
    "llama": [],
    "solar": []
}

llmevals = {
    "llama": [],
    "solar": []
}

# Load data
for llama_dir, mistral_dir, solar_dir in zip(llama_dirs, mistral_dirs, solar_dirs):
    with open(f"{llama_dir}/eval_dev_metrics.json", "r") as f:
        llama_metrics = json.load(f)
        recalls["llama"].append(llama_metrics.get("Recall", 0))
        llmevals["llama"].append(llama_metrics.get("LLMeval_vllm_SOLAR-107B", 0))

    with open(f"{solar_dir}/eval_dev_metrics.json", "r") as f:
        solar_metrics = json.load(f)
        recalls["solar"].append(solar_metrics.get("Recall", 0))
        llmevals["solar"].append(solar_metrics.get("LLMeval_vllm_SOLAR-107B", 0))

# Define plot parameters
x = np.arange(len(recalls["llama"]))
width = 0.4
# textures = ['/', '\\', '*']  # Textures for bars
textures = [None, None, None]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Plot Recall
ax1 = axes[0]
rects1 = ax1.bar(x - 0.2, recalls["llama"], width, label='LLAMA-1B', color='steelblue', hatch=textures[0])
rects3 = ax1.bar(x + 0.2, recalls["solar"], width, label='SOLAR-10.7B', color='salmon', hatch=textures[2])
ax1.set_ylabel('Recall')
ax1.set_title('Recall distractors')
ax1.set_xticks(x)
ax1.set_xticklabels(labels, rotation=90)
ax1.legend()
for rect, data_key in zip([rects1, rects3], ["llama", "solar"]):
    for i, rect_bar in enumerate(rect):
        ax1.text(rect_bar.get_x() + rect_bar.get_width() / 2.0, rect_bar.get_height(),
                 f"{recalls[data_key][i]:.3f}", ha='center', va='bottom', fontsize=6)

# Plot LLMeval
ax2 = axes[1]
rects4 = ax2.bar(x - 0.2, llmevals["llama"], width, label='LLAMA-1B', color='steelblue', hatch=textures[0])
rects6 = ax2.bar(x + 0.2, llmevals["solar"], width, label='SOLAR-10.7B', color='salmon', hatch=textures[2])
ax2.set_ylabel('LLMeval')
ax2.set_title('LLMeval distractors')
ax2.set_xticks(x)
ax2.set_xticklabels(labels, rotation=90)
ax2.legend()
for rect, data_key in zip([rects4, rects6], ["llama", "solar"]):
    for i, rect_bar in enumerate(rect):
        ax2.text(rect_bar.get_x() + rect_bar.get_width() / 2.0, rect_bar.get_height(),
                 f"{llmevals[data_key][i]:.3f}", ha='center', va='bottom', fontsize=6)

plt.tight_layout()
plt.savefig('figs/benchmarks_rerankers.png')
plt.show()